# 02 · Extracción financiera híbrida y robusta

## Objetivo

Este notebook procesa los archivos Excel de la carpeta `00_datos_crudos` y construye un dataset financiero en **formato largo** con los siguientes indicadores:

| Indicador final | Fuente principal | Estrategia de búsqueda |
|---|---|---|
| `activos_totales` | Balance | Búsqueda por etiqueta |
| `pasivos_totales` | Balance | Búsqueda por etiqueta |
| `patrimonio` | Balance | Búsqueda por etiqueta |
| `roe` | Indicadores | Código `IF295`; si falla, búsqueda por nombre/variantes |
| `morosidad` | Indicadores | Código `IF013`; si falla, búsqueda por nombre/variantes |
| `solvencia_proxy` | Indicadores | Código `IF112`; si falla, búsqueda por nombre/variantes; si no existe, cálculo `(FK / FI) * 100` con `IF41` e `IF111` |

## Reglas clave

1. La fecha del periodo se obtiene **únicamente desde la hoja Balance** o desde una hoja que contenga la cabecera `ESTADO DE SITUACIÓN`.
2. Para ROE, morosidad y solvencia_proxy proxy se busca primero por código. Si no existe código útil, se busca por nombre y variantes.
3. Si se encuentran filas duplicadas por código o por nombre, se usa la fila que tenga más valores reales bajo columnas de bancos.
4. Si un banco no existe en un periodo, **no se crea fila artificial** para ese banco.
5. Si un banco sí existe en el encabezado de ese periodo pero su celda está vacía o tiene error de Excel, se conserva la fila con `NaN`.
6. Se generan datasets auxiliares de bancos y periodos para alimentar filtros del dashboard.

Este notebook está pensado para ser una versión más robusta del flujo anterior, evitando nulos causados por cambios de nombres en los indicadores.

## 1. Dependencias

Ejecuta la instalación solo si tu ambiente todavía no tiene estas librerías.

```bash
pip install pandas numpy openpyxl xlrd pyarrow
```

- `openpyxl`: lectura de `.xlsx` y `.xlsm`.
- `xlrd`: lectura de `.xls`.
- `pyarrow`: exportación a Parquet.

In [2]:
# Descomenta esta línea solo si necesitas instalar dependencias desde el notebook.
!pip install pandas numpy openpyxl xlrd pyarrow

## 2. Importar librerías

Se importan librerías para leer Excel, recorrer carpetas, normalizar textos, detectar fechas, paralelizar el procesamiento y exportar resultados.

In [3]:
from pathlib import Path
from datetime import datetime, date
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
import os
import re
import time
import math
import unicodedata
import warnings

import numpy as np
import pandas as pd

try:
    from openpyxl.utils import get_column_letter
except Exception:
    get_column_letter = None

warnings.filterwarnings("ignore")

# Configuración visual para inspeccionar mejor tablas largas dentro del notebook.
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 160)
pd.set_option("display.width", 180)

## 3. Configuración de rutas

El notebook detecta automáticamente la raíz del proyecto buscando la carpeta `00_datos_crudos`.

Puede ejecutarse desde:

- la raíz del proyecto;
- `03_cuadernos`;
- una subcarpeta interna, siempre que exista `00_datos_crudos` en algún nivel superior.

In [4]:
def encontrar_raiz_proyecto(nombre_carpeta_datos="00_datos_crudos") -> Path:
    """Busca la raíz del proyecto subiendo desde la carpeta actual hasta encontrar 00_datos_crudos."""
    actual = Path.cwd().resolve()

    # Recorremos la carpeta actual y sus padres para que el notebook funcione desde varias ubicaciones.
    for candidato in [actual] + list(actual.parents):
        if (candidato / nombre_carpeta_datos).exists():
            return candidato

    raise FileNotFoundError(
        f"No se encontró la carpeta {nombre_carpeta_datos!r} desde {actual}. "
        "Verifica que el notebook esté dentro del proyecto."
    )


RAIZ_PROYECTO = encontrar_raiz_proyecto()
RAW_DIR = RAIZ_PROYECTO / "00_datos_crudos"
PROCESSED_DIR = RAIZ_PROYECTO / "01_datos_procesados"
AUDIT_DIR = PROCESSED_DIR / "auditoria"

# Creamos las carpetas de salida si todavía no existen.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", RAIZ_PROYECTO)
print("Carpeta de datos crudos:", RAW_DIR)
print("Carpeta de salida:", PROCESSED_DIR)
print("Carpeta de auditoría:", AUDIT_DIR)

Raíz del proyecto: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador
Carpeta de datos crudos: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\00_datos_crudos
Carpeta de salida: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados
Carpeta de auditoría: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\auditoria


## 4. Parámetros de extracción

Aquí se definen:

- extensiones de Excel permitidas;
- cantidad de hilos para acelerar el procesamiento;
- indicadores esperados;
- códigos y nombres alternativos.

> Recomendación: si tu equipo tiene poca memoria, usa `N_WORKERS = 1` o `N_WORKERS = 2`.

In [5]:
EXTENSIONES_EXCEL = {".xls", ".xlsx", ".xlsm"}

# =============================================================================
# Configuración de rendimiento
# =============================================================================
# Opciones disponibles:
# - "process": usa ProcessPoolExecutor. Suele ser más rápido para muchos Excel porque
#   reparte la lectura/parsing en varios procesos.
# - "thread": usa ThreadPoolExecutor. Es más estable en notebooks de Windows si
#   ProcessPoolExecutor falla por serialización.
# - "secuencial": procesa archivo por archivo. Es más lento, pero útil para depurar.
MODO_PROCESAMIENTO = "process"

# Número de trabajadores paralelos.
# Recomendación: no usar todos los núcleos para evitar saturar memoria y disco.
N_WORKERS = max(1, min(6, (os.cpu_count() or 2) - 1))

# Tamaño de lote para ProcessPoolExecutor.
# En lugar de enviar 205 tareas pequeñas, enviamos lotes de archivos. Esto reduce
# overhead entre procesos y normalmente acelera la ejecución.
TAMANIO_LOTE_PROCESS = 8

# Cada cuántos lotes/archivos se muestra progreso.
MOSTRAR_PROGRESO_CADA = 5

# Si se activa, también extrae agregados como TOTAL BANCOS PRIVADOS.
# Para el dataset por institución bancaria debe permanecer en False.
INCLUIR_AGREGADOS = False

# Si un indicador no aparece en la hoja de Indicadores resuelta, esta opción permite escanear otras hojas.
# Se deja en True para robustez, pero solo se activa cuando falta algún indicador.
ESCANEAR_HOJAS_EXTRA_SI_FALLA = True

# Número mínimo de valores no vacíos para considerar que una fila candidata realmente contiene datos.
# No obliga a que todos los bancos tengan valor; solo evita escoger títulos o secciones sin datos.
MIN_VALORES_NO_VACIOS_CANDIDATO = 1

# Indicadores contables del Balance. No suelen tener código IF, por eso se buscan por etiqueta.
INDICADORES_BALANCE = {
    "activos_totales": {
        "etiquetas": [
            "TOTAL ACTIVO",
            "TOTAL ACTIVOS",
            "TOTAL DEL ACTIVO",
        ],
        "unidad": "miles_usd",
        "sentido": "mayor_es_tamano",
        "descripcion": "Total de activos del banco",
    },
    "pasivos_totales": {
        "etiquetas": [
            "TOTAL PASIVO",
            "TOTAL PASIVOS",
            "TOTAL DEL PASIVO",
        ],
        "unidad": "miles_usd",
        "sentido": "informativo",
        "descripcion": "Total de pasivos del banco",
    },
    "patrimonio": {
        "etiquetas": [
            "TOTAL PATRIMONIO",
            "TOTAL DEL PATRIMONIO",
            "PATRIMONIO TOTAL",
        ],
        "unidad": "miles_usd",
        "sentido": "mayor_es_mejor",
        "descripcion": "Patrimonio total del banco",
    },
}

# Indicadores financieros de la hoja Indicadores.
# Regla: primero se busca por código. Si no hay fila útil por código, se usa búsqueda textual.
INDICADORES_FINANCIEROS = {
    "roe": {
        "codigos": ["IF295"],
        "nombres": [
            "RESULTADOS DEL EJERCICIO / PATRIMONIO PROMEDIO",
            "RESULTADO DEL EJERCICIO / PATRIMONIO PROMEDIO",
            "RESULTADOS DEL EJERCICIO PATRIMONIO PROMEDIO",
            "RESULTADO DEL EJERCICIO PATRIMONIO PROMEDIO",
            "RENTABILIDAD SOBRE PATRIMONIO",
            "RENDIMIENTO SOBRE PATRIMONIO",
            "RETORNO SOBRE PATRIMONIO",
            "ROE",
        ],
        "unidad": "porcentaje",
        "sentido": "mayor_es_mejor",
        "descripcion": "Rentabilidad sobre patrimonio promedio",
    },
    "morosidad": {
        "codigos": ["IF013"],
        "nombres": [
            "MOROSIDAD DE LA CARTERA TOTAL",
            "MOROSIDAD CARTERA TOTAL",
            "INDICE DE MOROSIDAD DE LA CARTERA TOTAL",
            "INDICE DE MOROSIDAD CARTERA TOTAL",
            "TASA DE MOROSIDAD DE LA CARTERA TOTAL",
            "TASA DE MOROSIDAD CARTERA TOTAL",
            "MOROSIDAD TOTAL",
        ],
        "unidad": "porcentaje",
        "sentido": "menor_es_mejor",
        "descripcion": "Morosidad de la cartera total",
    },
    "solvencia_proxy": {
        "codigos": ["IF112"],
        "nombres": [
            "INDICE DE CAPITALIZACION NETO: FK / FI",
            "ÍNDICE DE CAPITALIZACIÓN NETO: FK / FI",
            "INDICE DE CAPITALIZACION NETO FK FI",
            "INDICE DE CAPITALIZACION NETO",
            "CAPITALIZACION NETO FK FI",
            "CAPITALIZACION NETA FK FI",
            "FK / FI",
            "FK FI",
        ],
        "unidad": "porcentaje",
        "sentido": "mayor_es_mejor",
        "descripcion": "Índice de capitalización neto FK/FI usado como proxy de solvencia",
    },
}

# Componentes para calcular solvencia_proxy si no existe IF112 ni su nombre.
# Fórmula requerida:
# solvencia_proxy = (FK / FI) * 100
# FK = (PATRIMONIO + RESULTADOS - OTROS INGRESOS) / ACTIVOS TOTALES
# FI = 1 + (ACTIVOS IMPRODUCTIVOS / ACTIVOS TOTALES)
COMPONENTES_SOLVENCIA_PROXY = {
    "fk": {
        "codigos": ["IF41", "IF041"],
        "nombres": [
            "FK = (PATRIMONIO + RESULTADOS - OTROS INGRESOS) / ACTIVOS TOTALES",
            "FK=(PATRIMONIO + RESULTADOS - OTROS INGRESOS) / ACTIVOS TOTALES",
            "FK PATRIMONIO RESULTADOS OTROS INGRESOS ACTIVOS TOTALES",
            "PATRIMONIO + RESULTADOS - OTROS INGRESOS / ACTIVOS TOTALES",
            "PATRIMONIO MAS RESULTADOS MENOS OTROS INGRESOS ACTIVOS TOTALES",
        ],
        "descripcion": "FK = (Patrimonio + resultados - otros ingresos) / activos totales",
    },
    "fi": {
        "codigos": ["IF111"],
        "nombres": [
            "FI = 1 + (ACTIVOS IMPRODUCTIVOS / ACTIVOS TOTALES)",
            "FI=1 + (ACTIVOS IMPRODUCTIVOS / ACTIVOS TOTALES)",
            "FI 1 ACTIVOS IMPRODUCTIVOS ACTIVOS TOTALES",
            "1 + (ACTIVOS IMPRODUCTIVOS / ACTIVOS TOTALES)",
            "1 ACTIVOS IMPRODUCTIVOS ACTIVOS TOTALES",
        ],
        "descripcion": "FI = 1 + (Activos improductivos / activos totales)",
    },
}


# Dataset largo con trazabilidad. Se conserva internamente para auditoría.
COLUMNAS_DATASET_FINAL = [
    "periodo",
    "fecha_corte",
    "banco_original",
    "banco_estandarizado",
    "indicador",
    "valor",
    "unidad",
    "sentido",
    "archivo_origen",
    "hoja_origen",
    "celda_origen",
    "codigo_origen",
    "descripcion_origen",
    "metodo_busqueda",
    "fila_origen_excel",
    "columna_banco_excel",
]

# Dataset final limpio solicitado para análisis y dashboard.
COLUMNAS_DATASET_LIMPIO = [
    "periodo",
    "banco_estandarizado",
    "indicador",
    "valor",
    "unidad",
    "sentido",
]

print("Modo de procesamiento:", MODO_PROCESAMIENTO)
print("N_WORKERS:", N_WORKERS)
print("Tamaño de lote ProcessPool:", TAMANIO_LOTE_PROCESS)
print("Indicadores Balance:", list(INDICADORES_BALANCE.keys()))
print("Indicadores financieros:", {k: v["codigos"] for k, v in INDICADORES_FINANCIEROS.items()})

Modo de procesamiento: process
N_WORKERS: 6
Tamaño de lote ProcessPool: 8
Indicadores Balance: ['activos_totales', 'pasivos_totales', 'patrimonio']
Indicadores financieros: {'roe': ['IF295'], 'morosidad': ['IF013'], 'solvencia_proxy': ['IF112']}


## 5. Funciones de normalización

Estas funciones convierten textos heterogéneos a una forma comparable.

Ejemplos:

- `ÍNDICE DE CAPITALIZACIÓN NETO` → `INDICE DE CAPITALIZACION NETO`
- `IF 013` → `IF13`
- `IF-013` → `IF13`

In [6]:
def quitar_tildes(texto: str) -> str:
    """Elimina tildes y diacríticos para comparar textos sin depender de acentos."""
    if texto is None:
        return ""

    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto if not unicodedata.combining(c))


def normalizar_texto(valor) -> str:
    """Normaliza un valor de Excel para búsquedas generales."""
    if pd.isna(valor):
        return ""

    # Convertimos a mayúsculas, quitamos tildes y reemplazamos saltos de línea por espacios.
    texto = quitar_tildes(str(valor)).upper()
    texto = texto.replace("\n", " ").replace("\r", " ").replace("\t", " ")

    # Quitamos espacios duplicados para evitar falsos negativos.
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def normalizar_busqueda(valor) -> str:
    """Normaliza texto eliminando signos que suelen variar entre archivos."""
    texto = normalizar_texto(valor)

    # Los signos se reemplazan por espacio para que frases con '/', '-', ':' sigan siendo comparables.
    texto = re.sub(r"[^A-Z0-9 ]+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def normalizar_codigo_indicador(valor) -> str:
    """Normaliza códigos IF. IF013, IF 013 e IF-013 se vuelven IF13."""
    texto = normalizar_texto(valor)

    # Quitamos separadores comunes entre letras y números.
    texto = re.sub(r"[\s\-_.]+", "", texto)

    match = re.search(r"IF0*(\d+)", texto)
    if not match:
        return ""

    return f"IF{int(match.group(1))}"


def codigos_normalizados(codigos):
    """Devuelve un conjunto de códigos normalizados para comparación rápida."""
    return {normalizar_codigo_indicador(codigo) for codigo in codigos}


def patron_en_texto(texto_normalizado: str, patron: str) -> bool:
    """Valida si un patrón normalizado está contenido dentro de un texto normalizado."""
    patron_normalizado = normalizar_busqueda(patron)
    return bool(patron_normalizado and patron_normalizado in texto_normalizado)


for indicador, config in INDICADORES_FINANCIEROS.items():
    print(indicador, config["codigos"], "=>", codigos_normalizados(config["codigos"]))

roe ['IF295'] => {'IF295'}
morosidad ['IF013'] => {'IF13'}
solvencia_proxy ['IF112'] => {'IF112'}


## 6. Funciones para fechas

La fecha del periodo se buscará solamente en la hoja Balance resuelta.

Se soportan fechas como:

- `28-FEB-2015`
- `28/02/2015`
- `2015-02-28`
- `28 DE FEBRERO DE 2015`
- serial Excel como `42063`

In [7]:
MESES_ES = {
    "ENE": 1, "ENERO": 1,
    "FEB": 2, "FEBRERO": 2,
    "MAR": 3, "MARZO": 3,
    "ABR": 4, "ABRIL": 4,
    "MAY": 5, "MAYO": 5,
    "JUN": 6, "JUNIO": 6,
    "JUL": 7, "JULIO": 7,
    "AGO": 8, "AGOSTO": 8,
    "SEP": 9, "SEPT": 9, "SEPTIEMBRE": 9,
    "OCT": 10, "OCTUBRE": 10,
    "NOV": 11, "NOVIEMBRE": 11,
    "DIC": 12, "DICIEMBRE": 12,
}


def fecha_desde_serial_excel(valor):
    """Convierte un número serial de Excel a fecha si está dentro de un rango razonable."""
    try:
        if isinstance(valor, bool):
            return None

        numero = float(valor)

        # Este rango cubre fechas aproximadas entre 1982 y 2064, suficiente para el proyecto.
        if numero < 30000 or numero > 60000:
            return None

        fecha = pd.to_datetime(numero, unit="D", origin="1899-12-30").date()
        return fecha if 1990 <= fecha.year <= 2050 else None
    except Exception:
        return None


def parsear_fecha(valor):
    """Intenta convertir un valor de Excel a fecha."""
    if pd.isna(valor):
        return None

    if isinstance(valor, datetime):
        return valor.date()

    if isinstance(valor, date):
        return valor

    if isinstance(valor, (int, float, np.integer, np.floating)):
        return fecha_desde_serial_excel(valor)

    texto_original = str(valor).strip()
    if not texto_original:
        return None

    texto = normalizar_texto(texto_original).replace(",", " ")
    texto = re.sub(r"\s+", " ", texto).strip()

    # Formato yyyy-mm-dd o yyyy/mm/dd.
    match = re.search(r"(\d{4})[/-](\d{1,2})[/-](\d{1,2})", texto)
    if match:
        anio, mes, dia = map(int, match.groups())
        try:
            return date(anio, mes, dia)
        except ValueError:
            return None

    # Formato dd/mm/yyyy o dd-mm-yyyy.
    match = re.search(r"(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})", texto)
    if match:
        dia, mes, anio = map(int, match.groups())
        if anio < 100:
            anio += 2000
        try:
            return date(anio, mes, dia)
        except ValueError:
            return None

    # Formato 28-FEB-2015, 28 FEB 2015 o 28 DE FEBRERO DE 2015.
    texto_limpio = texto.replace(" DE ", " ")
    match = re.search(r"(\d{1,2})[\s/-]+([A-Z]+)[\s/-]+(\d{4})", texto_limpio)
    if match:
        dia = int(match.group(1))
        mes_txt = match.group(2)
        anio = int(match.group(3))
        mes = MESES_ES.get(mes_txt)
        if mes:
            try:
                return date(anio, mes, dia)
            except ValueError:
                return None

    return None


def detectar_fecha_en_balance(df_balance, max_filas=30, max_columnas=20):
    """Busca la fecha únicamente en la cabecera de la hoja Balance."""
    filas = min(max_filas, df_balance.shape[0])
    columnas = min(max_columnas, df_balance.shape[1])

    for i in range(filas):
        for j in range(columnas):
            fecha = parsear_fecha(df_balance.iat[i, j])

            if fecha is not None:
                return {
                    "fecha_corte": fecha.isoformat(),
                    "periodo": fecha.strftime("%Y-%m"),
                    "celda_fecha": celda_excel(i, j),
                    "valor_fecha_original": df_balance.iat[i, j],
                }

    return {
        "fecha_corte": None,
        "periodo": None,
        "celda_fecha": None,
        "valor_fecha_original": None,
    }

## 7. Funciones auxiliares de Excel y números

Estas funciones ayudan a:

- elegir el motor de lectura correcto;
- convertir índices de DataFrame a celdas Excel;
- limpiar valores numéricos;
- preservar `NaN` cuando un banco existe pero no reporta un valor.

In [8]:
def engine_excel(ruta_archivo: Path):
    """Selecciona el motor de lectura según la extensión del archivo."""
    ext = ruta_archivo.suffix.lower()

    if ext == ".xls":
        return "xlrd"

    if ext in {".xlsx", ".xlsm"}:
        return "openpyxl"

    return None


def celda_excel(fila_idx: int, col_idx: int) -> str:
    """Convierte índices base cero a coordenada Excel. Ejemplo: 0,0 -> A1."""
    if get_column_letter is not None:
        return f"{get_column_letter(col_idx + 1)}{fila_idx + 1}"

    letras = ""
    n = col_idx + 1
    while n:
        n, rem = divmod(n - 1, 26)
        letras = chr(65 + rem) + letras
    return f"{letras}{fila_idx + 1}"


def letra_columna_excel(col_idx: int) -> str:
    """Obtiene solo la letra de la columna Excel."""
    return re.sub(r"\d+", "", celda_excel(0, col_idx))


ERRORES_EXCEL = {
    "#DIV/0!", "#¡DIV/0!", "#N/A", "#NULO!", "#NULL!",
    "#VALUE!", "#VALOR!", "#REF!", "#NAME?", "#¿NOMBRE?"
}


def limpiar_valor_numerico(valor):
    """
    Convierte un valor de Excel a número.

    Importante:
    - Celda vacía -> NaN.
    - Error de Excel -> NaN.
    - Texto no numérico -> NaN.
    - Cero real -> 0.0.
    """
    if pd.isna(valor):
        return np.nan

    if isinstance(valor, bool):
        return np.nan

    if isinstance(valor, (int, float, np.integer, np.floating)):
        return float(valor)

    texto = str(valor).strip()
    if not texto:
        return np.nan

    if texto.upper() in ERRORES_EXCEL:
        return np.nan

    # Limpieza de símbolos frecuentes en valores financieros.
    texto = texto.replace("%", "")
    texto = texto.replace("$", "")
    texto = texto.replace(" ", "")

    # Si aparece coma y punto, se asume coma como separador de miles.
    if "," in texto and "." in texto:
        texto = texto.replace(",", "")
    # Si solo aparece coma, se asume coma decimal.
    elif "," in texto and "." not in texto:
        texto = texto.replace(",", ".")

    try:
        return float(texto)
    except Exception:
        return np.nan


# Indicadores que en Excel pueden estar almacenados como número decimal
# pero mostrarse visualmente con formato de porcentaje.
# Ejemplo:
# - Excel muestra: 8,95%
# - Valor interno leído por pandas: 0.0895
# - Valor esperado para el dataset: 8.95
INDICADORES_PORCENTUALES_EXCEL = {"roe", "morosidad", "solvencia_proxy"}


def normalizar_valor_porcentual_excel(valor, indicador):
    """
    Corrige valores porcentuales cuando Excel los almacena como decimales.

    En archivos Excel, una celda con formato porcentaje puede verse como 8,95%,
    pero internamente se almacena como 0.0895. Pandas lee el valor interno,
    por eso es necesario multiplicar por 100 para que el dataset conserve
    el porcentaje visible esperado.

    Reglas:
    - Solo aplica a roe, morosidad y solvencia_proxy.
    - Si el valor está entre -1 y 1, se asume que viene de formato porcentaje
      de Excel y se multiplica por 100.
    - Si el valor ya viene como 8.95, 12.40 o 100.00, se conserva igual.
    - NaN se conserva como NaN.
    """
    if str(indicador).strip().lower() not in INDICADORES_PORCENTUALES_EXCEL:
        return valor

    if pd.isna(valor):
        return np.nan

    try:
        valor_num = float(valor)
    except Exception:
        return np.nan

    # Excel almacena 8.95% como 0.0895.
    # También almacena 100% como 1.0, por eso se incluye el límite <= 1.
    if valor_num != 0 and abs(valor_num) <= 1:
        return valor_num * 100

    return valor_num


def valor_no_vacio(valor) -> bool:
    """Determina si una celda tiene algún contenido reportado, aunque luego no sea numérico."""
    if pd.isna(valor):
        return False

    texto = str(valor).strip()
    if not texto:
        return False

    if texto.upper() in ERRORES_EXCEL:
        return False

    return True

## 8. Detección y estandarización de bancos

El objetivo es extraer solamente bancos reales y no categorías agregadas.

Regla de negocio importante:

- Si un banco no aparece en el encabezado del archivo de un periodo, no se genera registro.
- Si aparece en el encabezado, se genera registro aunque su valor esté vacío; ese valor queda como `NaN`.
- Los nombres históricos o escritos con variantes se consolidan en un único `banco_estandarizado`.

Normalizaciones especiales incluidas en esta versión:

| Variante encontrada | Nombre estándar |
|---|---|
| `BPDINERS`, `BP DINERS`, `DINERS CLUB` | `DINERS` |
| `BP D MIRO S.A.`, `BP D-MIRO S.A.`, `BP D-MIRO` | `D-MIRO` |
| `BP FINCA`, `BP FINCA S.A.` | `FINCA` |
| `BP VISIONFUND ECUADOR`, `BP VISIONFUND ECUADOR S.A.` | `VISIONFUND ECUADOR` |
| `BP BANCO DESARROLLO DE LOS PUEBLOS S.A., CODESARROLLO` | `DESARROLLO DE LOS PUEBLOS S.A` |

In [9]:
# Lista referencial para reconocer bancos aunque no tengan prefijo BP.
# No reemplaza el nombre original; solo ayuda a clasificar encabezados.
BANCOS_REFERENCIA = [
    "PICHINCHA", "GUAYAQUIL", "PACIFICO", "PACÍFICO", "PRODUBANCO", "PRODUCCION", "PRODUCCIÓN",
    "AUSTRO", "BOLIVARIANO", "INTERNACIONAL", "MACHALA", "LOJA", "GENERAL RUMIÑAHUI", "RUMINAHUI",
    "SOLIDARIO", "DINERS", "BPDINERS", "AMAZONAS", "PROCREDIT", "LITORAL", "COMERCIAL DE MANABI", "MANABI",
    "CAPITAL", "FINCA", "COFIEC", "DELBANK", "D-MIRO", "D MIRO", "DMIRO", "DESARROLLO",
    "VISIONFUND", "VISIONFUND ECUADOR", "BANCOOP", "CODESARROLLO", "BANCODESARROLLO",
    "TERRITORIAL", "SUDAMERICANO", "UNIBANCO", "CITIBANK", "CITI", "ATLANTIDA", "ATLÁNTIDA",
    "BANISI", "DINERS CLUB",
]

CATEGORIAS_EXCLUIR = [
    "TOTAL", "BANCOS PRIVADOS", "GRANDES", "MEDIANOS", "PEQUENOS", "PEQUEÑOS",
    "COMERCIALES", "CONSUMO", "VIVIENDA", "MICROEMPRESA", "PUBLICOS", "PÚBLICOS",
    "SISTEMA", "SUBTOTAL", "PROMEDIO",
]

CABECERAS_NO_BANCO = {
    "CODIGO", "CÓDIGO", "CUENTA", "DESCRIPCION", "DESCRIPCIÓN", "INDICADOR",
    "INDICADORES", "VARIACION", "VARIACIÓN", "ABSOLUTA", "RELATIVA", "PERIODO", "FECHA",
    "CONCEPTO", "DETALLE", "SALDO", "PORCENTAJE", "RANKING", "POSICION", "POSICIÓN",
}

# Diccionario de nombres canónicos para bancos con variantes históricas o inconsistencias frecuentes.
# La clave se evalúa contra una versión compacta del nombre: sin espacios, sin tildes y sin signos.
BANCOS_CANONICOS_COMPACTOS = {
    "BPDINERS": "DINERS",
    "DINERS": "DINERS",
    "DINERSCLUB": "DINERS",
    "BPDINERSCLUB": "DINERS",
    "DMIRO": "D-MIRO",
    "BPDMIRO": "D-MIRO",
    "DMIROSA": "D-MIRO",
    "BPDMIROSA": "D-MIRO",
    "FINCA": "FINCA",
    "BPFINCA": "FINCA",
    "FINCASA": "FINCA",
    "BPFINCASA": "FINCA",
    "VISIONFUNDECUADOR": "VISIONFUND ECUADOR",
    "BPVISIONFUNDECUADOR": "VISIONFUND ECUADOR",
    "VISIONFUNDECUADORSA": "VISIONFUND ECUADOR",
    "BPVISIONFUNDECUADORSA": "VISIONFUND ECUADOR",
    "BANCODESARROLLO": "DESARROLLO DE LOS PUEBLOS S.A",
    "BPBANCODESARROLLO": "DESARROLLO DE LOS PUEBLOS S.A",
    "CODESARROLLO": "DESARROLLO DE LOS PUEBLOS S.A",
    "BPCODESARROLLO": "DESARROLLO DE LOS PUEBLOS S.A",
}


def compactar_nombre_banco(valor) -> str:
    """
    Crea una versión compacta del nombre del banco para comparar variantes.

    Ejemplo:
    - 'BP D-MIRO S.A.' -> 'BPDMIROSA'
    - 'BP FINCA S.A.' -> 'BPFINCASA'
    - 'BPDINERS' -> 'BPDINERS'
    """
    texto = normalizar_busqueda(valor)
    return re.sub(r"\s+", "", texto)


def quitar_prefijos_banco(texto: str) -> str:
    """
    Quita prefijos institucionales frecuentes.

    Se contempla el caso pegado 'BPDINERS', donde no existe espacio entre BP y el nombre.
    """
    texto = normalizar_busqueda(texto)

    # Si viene como BPDINERS, BPFINCA, BPVISIONFUND, etc., insertamos un espacio después de BP.
    texto = re.sub(r"^BP(?=[A-Z0-9])", "BP ", texto)

    # Quitamos prefijos repetibles al inicio del nombre.
    # Ejemplo: 'BP BANCO DESARROLLO...' -> 'DESARROLLO...'
    cambios = True
    while cambios:
        anterior = texto
        texto = re.sub(r"^BP\s+", "", texto).strip()
        texto = re.sub(r"^BANCO\s+", "", texto).strip()
        texto = re.sub(r"^BANCA\s+", "", texto).strip()
        cambios = texto != anterior

    return re.sub(r"\s+", " ", texto).strip()


def quitar_sufijos_societarios(texto: str) -> str:
    """
    Quita sufijos societarios cuando no forman parte del nombre canónico elegido.

    Ejemplo:
    - 'FINCA S A' -> 'FINCA'
    - 'VISIONFUND ECUADOR S A' -> 'VISIONFUND ECUADOR'
    """
    texto = normalizar_busqueda(texto)
    texto = re.sub(r"\bSOCIEDAD ANONIMA\b", "", texto)
    texto = re.sub(r"\bS A\b$", "", texto)
    texto = re.sub(r"\bSA\b$", "", texto)
    texto = re.sub(r"\bC A\b$", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def clasificar_entidad(nombre):
    """Clasifica una cabecera como banco, agregado o no válida."""
    texto = normalizar_busqueda(nombre)

    if not texto:
        return "sin_encabezado"

    if texto in CABECERAS_NO_BANCO:
        return "sin_encabezado"

    # Excluimos agregados para evitar que el dataset por banco incluya totales o grupos.
    if any(palabra in texto for palabra in map(normalizar_busqueda, CATEGORIAS_EXCLUIR)):
        return "agregado"

    # Muchos boletines usan BP como prefijo de Banco Privado.
    # También se acepta el caso pegado, por ejemplo: BPDINERS.
    if texto.startswith("BP ") or re.match(r"^BP[A-Z0-9]", texto):
        return "banco"

    # Fallback para casos donde el encabezado viene sin BP.
    if any(normalizar_busqueda(banco) in texto for banco in BANCOS_REFERENCIA):
        return "banco"

    return "sin_encabezado"


def estandarizar_nombre_banco(nombre):
    """
    Estandariza el nombre del banco sin destruir el nombre original.

    Esta función controla variantes históricas, prefijos, sufijos societarios y nombres pegados.
    El resultado se usa como clave maestra para evitar duplicados en:
    - dataset_financiero_limpio;
    - dataset_bancos;
    - filtros del dashboard.
    """
    if pd.isna(nombre):
        return ""

    # Primera comparación: nombre tal como viene, pero compactado.
    # Esto permite reconocer casos como BPDINERS antes de quitar prefijos.
    compacto_original = compactar_nombre_banco(nombre)
    if compacto_original in BANCOS_CANONICOS_COMPACTOS:
        return BANCOS_CANONICOS_COMPACTOS[compacto_original]

    # Quitamos prefijos como BP y BANCO.
    texto = quitar_prefijos_banco(nombre)

    # Segunda comparación: nombre sin prefijos, compactado.
    compacto_sin_prefijo = compactar_nombre_banco(texto)
    if compacto_sin_prefijo in BANCOS_CANONICOS_COMPACTOS:
        return BANCOS_CANONICOS_COMPACTOS[compacto_sin_prefijo]

    # Reglas especiales por contenido. Son útiles cuando el nombre tiene notas o descripciones adicionales.
    if "DESARROLLO DE LOS PUEBLOS" in texto or "CODESARROLLO" in texto or "BANCODESARROLLO" in compacto_sin_prefijo:
        return "DESARROLLO DE LOS PUEBLOS S.A"

    if "VISIONFUND ECUADOR" in texto or "VISIONFUND" in texto:
        return "VISIONFUND ECUADOR"

    if "D MIRO" in texto or "DMIRO" in compacto_sin_prefijo:
        return "D-MIRO"

    if "DINERS" in texto or "DINERS" in compacto_sin_prefijo:
        return "DINERS"

    if texto.startswith("FINCA") or compacto_sin_prefijo.startswith("FINCA"):
        return "FINCA"

    # Para el resto de bancos, quitamos sufijos como S.A. cuando solo generan duplicados.
    texto = quitar_sufijos_societarios(texto)

    # Normalizaciones puntuales útiles para unir variantes históricas.
    reemplazos = {
        "PACIFICO": "PACIFICO",
        "PACÍFICO": "PACIFICO",
        "PRODUCCION": "PRODUBANCO",
        "PRODUCCIÓN": "PRODUBANCO",
        "GENERAL RUMINAHUI": "GENERAL RUMIÑAHUI",
        "RUMINAHUI": "GENERAL RUMIÑAHUI",
        "COMERCIAL DE MANABI": "COMERCIAL DE MANABI",
        "MANABI": "COMERCIAL DE MANABI",
    }

    return reemplazos.get(texto, texto)


# Prueba rápida de las variantes reportadas.
pruebas_bancos = [
    "BPDINERS",
    "BP D MIRO S.A.",
    "BP D-MIRO S.A.",
    "BP D-MIRO",
    "BP FINCA",
    "BP FINCA S.A.",
    "BP VISIONFUND ECUADOR",
    "BP VISIONFUND ECUADOR S.A.",
    "BP BANCO  DESARROLLO DE LOS PUEBLOS  S.A., CODESARROLLO",
]

pd.DataFrame({
    "banco_original": pruebas_bancos,
    "banco_estandarizado": [estandarizar_nombre_banco(x) for x in pruebas_bancos],
})

,banco_original,banco_estandarizado
0,BPDINERS,DINERS
1,BP D MIRO S.A.,D-MIRO
2,BP D-MIRO S.A.,D-MIRO
3,BP D-MIRO,D-MIRO
4,BP FINCA,FINCA
5,BP FINCA S.A.,FINCA
6,BP VISIONFUND ECUADOR,VISIONFUND ECUADOR
7,BP VISIONFUND ECUADOR S.A.,VISIONFUND ECUADOR
8,"BP BANCO DESARROLLO DE LOS PUEBLOS S.A., COD...",DESARROLLO DE LOS PUEBLOS S.A


## 9. Resolver hojas Balance e Indicadores

La fecha se toma de Balance.

La hoja Balance se resuelve así:

1. Nombre exacto o similar a `BALANCE`.
2. Si no existe, búsqueda por contenido: `ESTADO DE SITUACIÓN`.

La hoja Indicadores se resuelve así:

1. Nombre que contenga `INDIC`.
2. Si no existe, búsqueda por códigos o nombres esperados en una vista previa.

In [10]:
def leer_preview(excel, hoja, nrows=100):
    """Lee pocas filas de una hoja para identificar su contenido sin cargar todo el archivo."""
    return pd.read_excel(excel, sheet_name=hoja, header=None, dtype=object, nrows=nrows)


def resolver_hoja_balance(excel):
    """Resuelve la hoja Balance usando nombre y, si hace falta, contenido."""
    hojas = excel.sheet_names

    # Primera opción: nombre de hoja similar a BALANCE.
    for hoja in hojas:
        nombre = normalizar_busqueda(hoja)
        if nombre == "BALANCE" or nombre.startswith("BALANCE ") or nombre.startswith("BALAN"):
            return hoja, "nombre_balance"

    # Segunda opción: buscar ESTADO DE SITUACION en el contenido de las primeras filas.
    for hoja in hojas:
        try:
            preview = leer_preview(excel, hoja, nrows=40)
            textos = " ".join(normalizar_busqueda(x) for x in preview.to_numpy().ravel() if not pd.isna(x))
            if "ESTADO DE SITUACION" in textos:
                return hoja, "contenido_estado_situacion"
        except Exception:
            continue

    return None, "no_encontrada"


def resolver_hojas_indicadores(excel):
    """Devuelve una lista priorizada de hojas candidatas para indicadores financieros."""
    hojas = excel.sheet_names
    candidatas = []

    # Primera opción: hojas cuyo nombre sea claramente INDICADORES o variante.
    for hoja in hojas:
        nombre = normalizar_busqueda(hoja)
        if "INDIC" in nombre:
            candidatas.append((hoja, "nombre_indicadores"))

    if candidatas:
        return candidatas

    # Segunda opción: buscar códigos o nombres esperados en una vista previa.
    codigos_objetivo = set()
    nombres_objetivo = []

    for config in INDICADORES_FINANCIEROS.values():
        codigos_objetivo.update(codigos_normalizados(config["codigos"]))
        nombres_objetivo.extend(config["nombres"])

    for hoja in hojas:
        try:
            preview = leer_preview(excel, hoja, nrows=100)
            plano = preview.to_numpy().ravel()

            contiene_codigo = any(normalizar_codigo_indicador(x) in codigos_objetivo for x in plano)
            contiene_nombre = any(
                any(patron_en_texto(normalizar_busqueda(x), patron) for patron in nombres_objetivo)
                for x in plano
                if not pd.isna(x)
            )

            if contiene_codigo or contiene_nombre:
                candidatas.append((hoja, "contenido_indicadores"))
        except Exception:
            continue

    return candidatas

## 10. Detección de encabezados oficiales de bancos

Para evitar que el proceso tome nombres de bancos desde filas incorrectas, los encabezados de bancos se obtienen únicamente desde filas oficiales:

- En la hoja de Balance: desde la fila que contiene la celda `CUENTA`.
- En la hoja de Indicadores: desde la fila que contiene la celda `NOMBRE DEL INDICADOR`.

Con esto se evita que textos internos, notas, categorías o filas descriptivas sean interpretadas como bancos.


In [11]:
# =============================================================================
# Encabezados oficiales para obtener nombres de bancos
# =============================================================================
# Regla solicitada:
# - En Balance, los bancos deben salir únicamente de la fila donde exista la celda "CUENTA".
# - En Indicadores, los bancos deben salir únicamente de la fila donde exista la celda "NOMBRE DEL INDICADOR".
# Esta regla evita tomar nombres desde filas intermedias, notas, categorías o textos descriptivos.

MARCADORES_ENCABEZADO_BANCOS = {
    "balance": {"CUENTA"},
    "indicadores": {"NOMBRE DEL INDICADOR", "NOMBRE DE INDICADOR", "NOMBRE INDICADOR"},
}


def tipo_marcador_encabezado_bancos(valor):
    """
    Identifica si una celda corresponde al marcador oficial de encabezado de bancos.

    Retorna:
    - "balance" si la celda dice CUENTA.
    - "indicadores" si la celda dice NOMBRE DEL INDICADOR.
    - None si la celda no es un marcador válido.
    """
    texto = normalizar_busqueda(valor)

    if not texto:
        return None

    # Marcador oficial para la hoja Balance.
    # Se usa igualdad exacta para no confundirlo con nombres de cuentas contables.
    if texto in MARCADORES_ENCABEZADO_BANCOS["balance"]:
        return "balance"

    # Marcador oficial para la hoja Indicadores.
    # Se contempla una variante corta por si algún boletín omite la palabra "DEL".
    if texto in MARCADORES_ENCABEZADO_BANCOS["indicadores"]:
        return "indicadores"

    return None


def contar_bancos_en_fila_oficial(df, fila_idx, col_marcador_idx, incluir_agregados=False):
    """
    Cuenta bancos válidos en una fila oficial de encabezado.

    La lectura empieza a la derecha de la celda marcador:
    - después de CUENTA;
    - después de NOMBRE DEL INDICADOR.

    Esto evita evaluar columnas previas que contienen códigos, descripciones u otros textos.
    """
    bancos = 0
    agregados = 0

    valores_fila = df.iloc[fila_idx].tolist()

    for col in range(col_marcador_idx + 1, len(valores_fila)):
        tipo = clasificar_entidad(valores_fila[col])

        if tipo == "banco":
            bancos += 1
        elif tipo == "agregado":
            agregados += 1

    if incluir_agregados:
        total_util = bancos + agregados
    else:
        total_util = bancos

    return bancos, agregados, total_util


def buscar_filas_encabezado_oficiales(df, fila_indicador_idx=None, incluir_agregados=False):
    """
    Busca filas oficiales de encabezado antes de la fila del indicador.

    Importante:
    - No se buscan bancos en cualquier fila.
    - Solo se aceptan filas que tengan el marcador oficial CUENTA o NOMBRE DEL INDICADOR.
    - Si hay más de una fila oficial, se prioriza la que tenga más bancos válidos.
    """
    filas_encontradas = []

    # El encabezado debe estar antes de la fila del indicador.
    # Si no se recibe fila_indicador_idx, se revisa toda la hoja.
    limite_filas = df.shape[0] if fila_indicador_idx is None else min(df.shape[0], max(0, fila_indicador_idx))

    for fila_idx in range(limite_filas):
        valores_fila = df.iloc[fila_idx].tolist()

        for col_idx, valor in enumerate(valores_fila):
            tipo_marcador = tipo_marcador_encabezado_bancos(valor)

            if tipo_marcador is None:
                continue

            bancos, agregados, total_util = contar_bancos_en_fila_oficial(
                df=df,
                fila_idx=fila_idx,
                col_marcador_idx=col_idx,
                incluir_agregados=incluir_agregados,
            )

            # Si la fila oficial no tiene bancos a la derecha, no sirve para extracción.
            if total_util == 0:
                continue

            filas_encontradas.append({
                "fila_encabezado_idx": fila_idx,
                "fila_encabezado_excel": fila_idx + 1,
                "col_marcador_idx": col_idx,
                "celda_marcador": celda_excel(fila_idx, col_idx),
                "marcador": valor,
                "tipo_marcador": tipo_marcador,
                "bancos_detectados": bancos,
                "agregados_detectados": agregados,
                "score": bancos * 10 + agregados,
            })

    return filas_encontradas


def encontrar_fila_encabezados_bancos(df, fila_indicador_idx, col_referencia_idx=0, ventana=140):
    """
    Devuelve la fila oficial desde donde se deben leer los nombres de bancos.

    Esta versión ya no usa búsqueda heurística por cualquier fila con textos parecidos a bancos.
    Solo acepta encabezados oficiales:
    - fila con CUENTA para Balance;
    - fila con NOMBRE DEL INDICADOR para Indicadores.

    El parámetro ventana se conserva por compatibilidad con el resto del notebook,
    pero ya no se usa para buscar encabezados en filas arbitrarias.
    """
    filas_oficiales = buscar_filas_encabezado_oficiales(
        df=df,
        fila_indicador_idx=fila_indicador_idx,
        incluir_agregados=INCLUIR_AGREGADOS,
    )

    if not filas_oficiales:
        return None

    # Se prioriza la fila oficial con más bancos detectados.
    # Como desempate, se toma la fila más cercana al indicador porque algunos archivos
    # pueden repetir encabezados en secciones inferiores.
    filas_ordenadas = sorted(
        filas_oficiales,
        key=lambda x: (
            x["score"],
            x["bancos_detectados"],
            x["agregados_detectados"],
            x["fila_encabezado_idx"],
        ),
        reverse=True,
    )

    return filas_ordenadas[0]


def obtener_columnas_bancos_desde_header(df, fila_header_idx, incluir_agregados=False):
    """
    Devuelve las columnas que corresponden a bancos existentes en esa hoja/periodo.

    La función recibe una fila_header_idx que ya fue validada como encabezado oficial:
    - fila con CUENTA;
    - fila con NOMBRE DEL INDICADOR.

    Por tanto, los bancos se leen únicamente desde esa fila oficial.
    """
    columnas = []

    for col in range(df.shape[1]):
        nombre = df.iat[fila_header_idx, col]
        tipo = clasificar_entidad(nombre)

        if tipo == "banco" or (incluir_agregados and tipo == "agregado"):
            columnas.append({
                "col_idx": col,
                "banco_original": nombre,
                "banco_estandarizado": estandarizar_nombre_banco(nombre),
                "tipo_entidad": tipo,
                "columna_excel": letra_columna_excel(col),
            })

    return columnas


def describir_fila(df, fila_idx, col_idx, ancho=4):
    """Obtiene una descripción corta del indicador desde celdas cercanas a la coincidencia."""
    textos = []

    for col in range(max(0, col_idx), min(df.shape[1], col_idx + ancho)):
        valor = df.iat[fila_idx, col]
        if pd.isna(valor):
            continue

        texto = str(valor).strip()
        if texto and pd.isna(limpiar_valor_numerico(texto)):
            textos.append(texto)

    return " | ".join(textos) if textos else None


## 11. Búsqueda de filas candidatas

La lógica principal es:

1. Buscar por código.
2. Evaluar si la fila tiene valores bajo columnas de bancos.
3. Si no hay código útil, buscar por nombre y variantes.
4. Si hay duplicados, escoger la fila con más valores.

In [12]:
def buscar_candidatos_por_codigo(df, codigos_objetivo):
    """Busca códigos IF dentro de toda la hoja."""
    codigos_norm = codigos_normalizados(codigos_objetivo)
    candidatos = []

    for i, fila in enumerate(df.itertuples(index=False, name=None)):
        for j, valor in enumerate(fila):
            codigo = normalizar_codigo_indicador(valor)
            if codigo in codigos_norm:
                candidatos.append({
                    "fila_idx": i,
                    "col_idx": j,
                    "celda": celda_excel(i, j),
                    "metodo_busqueda": "codigo",
                    "codigo_origen": valor,
                    "texto_origen": valor,
                })

    return candidatos


def buscar_candidatos_por_nombre(df, nombres_objetivo):
    """Busca nombres o variantes textuales dentro de toda la hoja."""
    candidatos = []

    for i, fila in enumerate(df.itertuples(index=False, name=None)):
        for j, valor in enumerate(fila):
            if pd.isna(valor):
                continue

            texto_norm = normalizar_busqueda(valor)
            if not texto_norm:
                continue

            if any(patron_en_texto(texto_norm, patron) for patron in nombres_objetivo):
                candidatos.append({
                    "fila_idx": i,
                    "col_idx": j,
                    "celda": celda_excel(i, j),
                    "metodo_busqueda": "nombre",
                    "codigo_origen": None,
                    "texto_origen": valor,
                })

    return candidatos


def buscar_candidatos_por_etiqueta(df, etiquetas_objetivo):
    """Busca etiquetas contables del Balance, como TOTAL ACTIVO o TOTAL PASIVO."""
    candidatos = []
    etiquetas_norm = [normalizar_busqueda(etiqueta) for etiqueta in etiquetas_objetivo]

    for i, fila in enumerate(df.itertuples(index=False, name=None)):
        for j, valor in enumerate(fila):
            if pd.isna(valor):
                continue

            texto_norm = normalizar_busqueda(valor)
            if texto_norm in etiquetas_norm:
                candidatos.append({
                    "fila_idx": i,
                    "col_idx": j,
                    "celda": celda_excel(i, j),
                    "metodo_busqueda": "etiqueta_balance",
                    "codigo_origen": None,
                    "texto_origen": valor,
                })

    return candidatos


def evaluar_candidato(df, candidato, incluir_agregados=False, ventana_header=140):
    """Evalúa si una fila candidata tiene encabezados de bancos y valores asociados."""
    fila_idx = candidato["fila_idx"]
    col_idx = candidato["col_idx"]

    header = encontrar_fila_encabezados_bancos(df, fila_idx, col_idx, ventana=ventana_header)
    if header is None:
        return {
            **candidato,
            "es_util": False,
            "motivo": "sin_encabezado_bancos",
            "fila_encabezado_idx": None,
            "bancos_detectados": 0,
            "valores_no_vacios": 0,
            "valores_numericos": 0,
            "score": -1,
        }

    columnas_bancos = obtener_columnas_bancos_desde_header(
        df,
        header["fila_encabezado_idx"],
        incluir_agregados=incluir_agregados,
    )

    valores_no_vacios = 0
    valores_numericos = 0

    for banco in columnas_bancos:
        col = banco["col_idx"]
        valor = df.iat[fila_idx, col] if col < df.shape[1] else np.nan

        if valor_no_vacio(valor):
            valores_no_vacios += 1

        if not pd.isna(limpiar_valor_numerico(valor)):
            valores_numericos += 1

    # Score diseñado para resolver duplicados: primero valores numéricos, luego no vacíos, luego bancos detectados.
    score = valores_numericos * 1000 + valores_no_vacios * 100 + len(columnas_bancos)

    return {
        **candidato,
        "es_util": valores_no_vacios >= MIN_VALORES_NO_VACIOS_CANDIDATO and len(columnas_bancos) > 0,
        "motivo": "ok" if valores_no_vacios >= MIN_VALORES_NO_VACIOS_CANDIDATO else "sin_valores_en_fila",
        "fila_encabezado_idx": header["fila_encabezado_idx"],
        "fila_encabezado_excel": header["fila_encabezado_excel"],
        "bancos_detectados": len(columnas_bancos),
        "valores_no_vacios": valores_no_vacios,
        "valores_numericos": valores_numericos,
        "score": score,
    }


def seleccionar_mejor_candidato(df, candidatos, incluir_agregados=False, ventana_header=140):
    """Selecciona la fila con más valores reales entre candidatos duplicados."""
    if not candidatos:
        return None, []

    evaluados = [evaluar_candidato(df, candidato, incluir_agregados, ventana_header) for candidato in candidatos]

    # Orden: útiles primero, mayor score después, y como desempate la fila más baja.
    evaluados_ordenados = sorted(
        evaluados,
        key=lambda x: (x["es_util"], x["score"], x["fila_idx"]),
        reverse=True,
    )

    return evaluados_ordenados[0], evaluados_ordenados

## 12. Extraer registros desde una fila elegida

Esta función crea las filas del dataset largo.

La extracción se hace solo sobre las columnas de bancos detectadas en la hoja del periodo.
Por eso, si un banco no aparece en un periodo, no se lo agrega artificialmente.

In [13]:
def construir_registros_desde_candidato(
    df,
    candidato,
    indicador,
    config,
    archivo_relativo,
    hoja,
    periodo,
    fecha_corte,
    incluir_agregados=False,
):
    """Convierte una fila de indicador en registros largos por banco."""
    registros = []

    fila_idx = candidato["fila_idx"]
    fila_header_idx = candidato["fila_encabezado_idx"]
    columnas_bancos = obtener_columnas_bancos_desde_header(df, fila_header_idx, incluir_agregados)
    descripcion_origen = describir_fila(df, fila_idx, candidato["col_idx"])

    for banco in columnas_bancos:
        col = banco["col_idx"]

        # Si el banco existe en el encabezado pero la celda está vacía, el valor queda NaN.
        valor_original = df.iat[fila_idx, col] if col < df.shape[1] else np.nan
        valor = limpiar_valor_numerico(valor_original)

        # Para ROE, morosidad y solvencia_proxy se corrige el caso típico de Excel:
        # si la celda se ve como 8,95%, pandas puede leer 0.0895. En ese caso
        # se transforma a 8.95 para conservar el porcentaje visible esperado.
        valor = normalizar_valor_porcentual_excel(valor, indicador)

        registros.append({
            "periodo": periodo,
            "fecha_corte": fecha_corte,
            "banco_original": banco["banco_original"],
            "banco_estandarizado": banco["banco_estandarizado"],
            "indicador": indicador,
            "valor": valor,
            "unidad": config["unidad"],
            "sentido": config["sentido"],
            "archivo_origen": archivo_relativo,
            "hoja_origen": hoja,
            "celda_origen": celda_excel(fila_idx, col),
            "codigo_origen": candidato.get("codigo_origen"),
            "descripcion_origen": descripcion_origen,
            "metodo_busqueda": candidato.get("metodo_busqueda"),
            "fila_origen_excel": fila_idx + 1,
            "columna_banco_excel": banco["columna_excel"],
        })

    return registros

## 13. Resolver indicadores dentro de una hoja

Esta función encapsula la regla híbrida:

- buscar por código;
- si no hay código útil, buscar por nombre;
- resolver duplicados por mayor cantidad de valores.

In [14]:
def resolver_indicador_financiero_en_hoja(df, indicador, config, incluir_agregados=False):
    """Busca un indicador financiero en una hoja usando código primero y nombre como respaldo."""
    eventos = []

    # 1. Búsqueda principal por código.
    candidatos_codigo = buscar_candidatos_por_codigo(df, config["codigos"])
    mejor_codigo, evaluados_codigo = seleccionar_mejor_candidato(df, candidatos_codigo, incluir_agregados)

    if mejor_codigo and mejor_codigo["es_util"]:
        if len(evaluados_codigo) > 1:
            eventos.append({
                "estado": "duplicado_codigo_resuelto",
                "indicador": indicador,
                "detalle": f"Se encontraron {len(evaluados_codigo)} coincidencias por código; se usó la fila con más valores.",
                "candidatos": [(e["celda"], e["valores_no_vacios"], e["valores_numericos"]) for e in evaluados_codigo],
            })
        return mejor_codigo, eventos

    # 2. Búsqueda secundaria por nombre si el código no existe o no tiene valores útiles.
    candidatos_nombre = buscar_candidatos_por_nombre(df, config["nombres"])
    mejor_nombre, evaluados_nombre = seleccionar_mejor_candidato(df, candidatos_nombre, incluir_agregados)

    if mejor_nombre and mejor_nombre["es_util"]:
        estado = "fallback_nombre_usado"
        if mejor_codigo and not mejor_codigo["es_util"]:
            estado = "codigo_sin_valores_fallback_nombre_usado"

        eventos.append({
            "estado": estado,
            "indicador": indicador,
            "detalle": "No se encontró una fila útil por código; se usó búsqueda textual por nombre/variantes.",
            "candidatos_codigo": [(e["celda"], e["motivo"], e["valores_no_vacios"]) for e in evaluados_codigo],
            "candidatos_nombre": [(e["celda"], e["valores_no_vacios"], e["valores_numericos"]) for e in evaluados_nombre],
        })

        if len(evaluados_nombre) > 1:
            eventos.append({
                "estado": "duplicado_nombre_resuelto",
                "indicador": indicador,
                "detalle": f"Se encontraron {len(evaluados_nombre)} coincidencias por nombre; se usó la fila con más valores.",
                "candidatos": [(e["celda"], e["valores_no_vacios"], e["valores_numericos"]) for e in evaluados_nombre],
            })

        return mejor_nombre, eventos

    # 3. Sin resultado útil.
    detalle = "No se encontró código ni nombre útil con valores bajo columnas de bancos."
    if candidatos_codigo or candidatos_nombre:
        detalle = "Se encontraron coincidencias, pero ninguna fila tenía valores útiles bajo columnas de bancos."

    eventos.append({
        "estado": "indicador_no_encontrado",
        "indicador": indicador,
        "detalle": detalle,
        "candidatos_codigo": [(e["celda"], e["motivo"], e["valores_no_vacios"]) for e in evaluados_codigo],
        "candidatos_nombre": [(e["celda"], e["motivo"], e["valores_no_vacios"]) for e in evaluados_nombre],
    })

    return None, eventos


def resolver_indicador_balance_en_hoja(df_balance, indicador, config, incluir_agregados=False):
    """Busca un indicador contable de Balance por etiqueta y resuelve duplicados por valores."""
    candidatos = buscar_candidatos_por_etiqueta(df_balance, config["etiquetas"])
    mejor, evaluados = seleccionar_mejor_candidato(df_balance, candidatos, incluir_agregados, ventana_header=60)

    eventos = []

    if mejor and mejor["es_util"]:
        if len(evaluados) > 1:
            eventos.append({
                "estado": "duplicado_balance_resuelto",
                "indicador": indicador,
                "detalle": f"Se encontraron {len(evaluados)} coincidencias; se usó la fila con más valores.",
                "candidatos": [(e["celda"], e["valores_no_vacios"], e["valores_numericos"]) for e in evaluados],
            })
        return mejor, eventos

    eventos.append({
        "estado": "indicador_balance_no_encontrado",
        "indicador": indicador,
        "detalle": "No se encontró una fila útil en Balance para este indicador.",
        "candidatos": [(e["celda"], e["motivo"], e["valores_no_vacios"]) for e in evaluados],
    })

    return None, eventos


# =============================================================================
# Solvencia proxy: búsqueda directa IF112 o cálculo por componentes FK/FI
# =============================================================================
def resolver_componente_solvencia_proxy_en_hoja(df, nombre_componente, config_componente, incluir_agregados=False):
    """
    Resuelve una fila componente de solvencia proxy.

    Orden de búsqueda:
    1. Código estándar del componente.
    2. Nombre o variantes textuales.
    3. Si hay duplicados, se conserva la fila con más valores bajo bancos.
    """
    eventos = []

    candidatos_codigo = buscar_candidatos_por_codigo(df, config_componente["codigos"])
    mejor_codigo, evaluados_codigo = seleccionar_mejor_candidato(df, candidatos_codigo, incluir_agregados)

    if mejor_codigo and mejor_codigo["es_util"]:
        if len(evaluados_codigo) > 1:
            eventos.append({
                "estado": f"duplicado_{nombre_componente}_codigo_resuelto",
                "indicador": "solvencia_proxy",
                "detalle": f"Se encontraron {len(evaluados_codigo)} coincidencias por código para {nombre_componente}; se usó la fila con más valores.",
                "candidatos": [(e["celda"], e["valores_no_vacios"], e["valores_numericos"]) for e in evaluados_codigo],
            })
        return mejor_codigo, eventos

    candidatos_nombre = buscar_candidatos_por_nombre(df, config_componente["nombres"])
    mejor_nombre, evaluados_nombre = seleccionar_mejor_candidato(df, candidatos_nombre, incluir_agregados)

    if mejor_nombre and mejor_nombre["es_util"]:
        eventos.append({
            "estado": f"fallback_{nombre_componente}_nombre_usado",
            "indicador": "solvencia_proxy",
            "detalle": f"No se encontró una fila útil por código para {nombre_componente}; se usó búsqueda textual por nombre/variantes.",
            "candidatos_codigo": [(e["celda"], e["motivo"], e["valores_no_vacios"]) for e in evaluados_codigo],
            "candidatos_nombre": [(e["celda"], e["valores_no_vacios"], e["valores_numericos"]) for e in evaluados_nombre],
        })

        if len(evaluados_nombre) > 1:
            eventos.append({
                "estado": f"duplicado_{nombre_componente}_nombre_resuelto",
                "indicador": "solvencia_proxy",
                "detalle": f"Se encontraron {len(evaluados_nombre)} coincidencias por nombre para {nombre_componente}; se usó la fila con más valores.",
                "candidatos": [(e["celda"], e["valores_no_vacios"], e["valores_numericos"]) for e in evaluados_nombre],
            })

        return mejor_nombre, eventos

    eventos.append({
        "estado": f"componente_{nombre_componente}_no_encontrado",
        "indicador": "solvencia_proxy",
        "detalle": f"No se encontró una fila útil para el componente {nombre_componente}.",
        "candidatos_codigo": [(e["celda"], e["motivo"], e["valores_no_vacios"]) for e in evaluados_codigo],
        "candidatos_nombre": [(e["celda"], e["motivo"], e["valores_no_vacios"]) for e in evaluados_nombre],
    })

    return None, eventos


def resolver_solvencia_proxy_en_hoja(df, config, incluir_agregados=False):
    """
    Resuelve solvencia_proxy en una hoja de Indicadores.

    Prioridad:
    1. Buscar IF112.
    2. Buscar por nombre: INDICE DE CAPITALIZACION NETO: FK / FI.
    3. Si no existe una fila directa útil, calcular con componentes:
       (IF41 / IF111) * 100.
    """
    eventos = []

    # 1. Búsqueda directa por código IF112.
    candidatos_codigo = buscar_candidatos_por_codigo(df, config["codigos"])
    mejor_codigo, evaluados_codigo = seleccionar_mejor_candidato(df, candidatos_codigo, incluir_agregados)

    if mejor_codigo and mejor_codigo["es_util"]:
        if len(evaluados_codigo) > 1:
            eventos.append({
                "estado": "duplicado_codigo_resuelto",
                "indicador": "solvencia_proxy",
                "detalle": f"Se encontraron {len(evaluados_codigo)} coincidencias por código IF112; se usó la fila con más valores.",
                "candidatos": [(e["celda"], e["valores_no_vacios"], e["valores_numericos"]) for e in evaluados_codigo],
            })

        return {
            "tipo": "directo",
            "candidato": mejor_codigo,
            "metodo_resolucion": "codigo_if112",
        }, eventos

    # 2. Búsqueda directa por nombre del índice de capitalización neto.
    candidatos_nombre = buscar_candidatos_por_nombre(df, config["nombres"])
    mejor_nombre, evaluados_nombre = seleccionar_mejor_candidato(df, candidatos_nombre, incluir_agregados)

    if mejor_nombre and mejor_nombre["es_util"]:
        estado = "fallback_nombre_usado"
        if mejor_codigo and not mejor_codigo["es_util"]:
            estado = "codigo_sin_valores_fallback_nombre_usado"

        eventos.append({
            "estado": estado,
            "indicador": "solvencia_proxy",
            "detalle": "No se encontró una fila útil por IF112; se usó el nombre del Índice de Capitalización Neto FK/FI.",
            "candidatos_codigo": [(e["celda"], e["motivo"], e["valores_no_vacios"]) for e in evaluados_codigo],
            "candidatos_nombre": [(e["celda"], e["valores_no_vacios"], e["valores_numericos"]) for e in evaluados_nombre],
        })

        if len(evaluados_nombre) > 1:
            eventos.append({
                "estado": "duplicado_nombre_resuelto",
                "indicador": "solvencia_proxy",
                "detalle": f"Se encontraron {len(evaluados_nombre)} coincidencias por nombre; se usó la fila con más valores.",
                "candidatos": [(e["celda"], e["valores_no_vacios"], e["valores_numericos"]) for e in evaluados_nombre],
            })

        return {
            "tipo": "directo",
            "candidato": mejor_nombre,
            "metodo_resolucion": "nombre_indice_capitalizacion",
        }, eventos

    # 3. Cálculo por componentes FK y FI.
    candidato_fk, eventos_fk = resolver_componente_solvencia_proxy_en_hoja(
        df=df,
        nombre_componente="fk",
        config_componente=COMPONENTES_SOLVENCIA_PROXY["fk"],
        incluir_agregados=incluir_agregados,
    )

    candidato_fi, eventos_fi = resolver_componente_solvencia_proxy_en_hoja(
        df=df,
        nombre_componente="fi",
        config_componente=COMPONENTES_SOLVENCIA_PROXY["fi"],
        incluir_agregados=incluir_agregados,
    )

    eventos.extend(eventos_fk)
    eventos.extend(eventos_fi)

    if candidato_fk and candidato_fk["es_util"] and candidato_fi and candidato_fi["es_util"]:
        eventos.append({
            "estado": "solvencia_proxy_calculada_fk_fi",
            "indicador": "solvencia_proxy",
            "detalle": "No se encontró IF112 ni el nombre directo. Se calculó solvencia_proxy con (FK / FI) * 100.",
            "candidatos": [
                ("FK", candidato_fk.get("celda"), candidato_fk.get("valores_no_vacios"), candidato_fk.get("valores_numericos")),
                ("FI", candidato_fi.get("celda"), candidato_fi.get("valores_no_vacios"), candidato_fi.get("valores_numericos")),
            ],
        })

        return {
            "tipo": "componentes_fk_fi",
            "candidato_fk": candidato_fk,
            "candidato_fi": candidato_fi,
            "metodo_resolucion": "calculo_fk_fi",
        }, eventos

    # 4. Sin resultado útil.
    detalle = (
        "No se encontró IF112 ni nombre directo útil. "
        "Tampoco fue posible calcular con FK/FI porque falta uno o ambos componentes."
    )

    eventos.append({
        "estado": "solvencia_proxy_no_encontrada",
        "indicador": "solvencia_proxy",
        "detalle": detalle,
        "candidatos_codigo": [(e["celda"], e["motivo"], e["valores_no_vacios"]) for e in evaluados_codigo],
        "candidatos_nombre": [(e["celda"], e["motivo"], e["valores_no_vacios"]) for e in evaluados_nombre],
    })

    return None, eventos


def construir_registros_solvencia_proxy_desde_componentes(
    df,
    candidato_fk,
    candidato_fi,
    config,
    archivo_relativo,
    hoja,
    periodo,
    fecha_corte,
    incluir_agregados=False,
):
    """
    Construye registros de solvencia_proxy calculando (FK / FI) * 100 por banco.

    La fila FK se usa como referencia principal de bancos. Para FI se busca la
    columna equivalente por banco estandarizado; si no se encuentra, se usa la
    misma columna como respaldo.
    """
    registros = []

    fila_fk_idx = candidato_fk["fila_idx"]
    fila_fi_idx = candidato_fi["fila_idx"]

    columnas_fk = obtener_columnas_bancos_desde_header(
        df,
        candidato_fk["fila_encabezado_idx"],
        incluir_agregados=incluir_agregados,
    )

    columnas_fi = obtener_columnas_bancos_desde_header(
        df,
        candidato_fi["fila_encabezado_idx"],
        incluir_agregados=incluir_agregados,
    )

    columnas_fi_por_banco = {
        banco["banco_estandarizado"]: banco
        for banco in columnas_fi
    }

    descripcion_fk = describir_fila(df, fila_fk_idx, candidato_fk["col_idx"])
    descripcion_fi = describir_fila(df, fila_fi_idx, candidato_fi["col_idx"])
    descripcion_origen = f"Cálculo (FK / FI) * 100 | FK: {descripcion_fk} | FI: {descripcion_fi}"

    for banco_fk in columnas_fk:
        banco_std = banco_fk["banco_estandarizado"]
        col_fk = banco_fk["col_idx"]

        banco_fi = columnas_fi_por_banco.get(banco_std)
        col_fi = banco_fi["col_idx"] if banco_fi else col_fk

        valor_fk = limpiar_valor_numerico(df.iat[fila_fk_idx, col_fk] if col_fk < df.shape[1] else np.nan)
        valor_fi = limpiar_valor_numerico(df.iat[fila_fi_idx, col_fi] if col_fi < df.shape[1] else np.nan)

        if pd.isna(valor_fk) or pd.isna(valor_fi) or valor_fi == 0:
            valor_calculado = np.nan
        else:
            valor_calculado = (valor_fk / valor_fi) * 100

        registros.append({
            "periodo": periodo,
            "fecha_corte": fecha_corte,
            "banco_original": banco_fk["banco_original"],
            "banco_estandarizado": banco_std,
            "indicador": "solvencia_proxy",
            "valor": valor_calculado,
            "unidad": config["unidad"],
            "sentido": config["sentido"],
            "archivo_origen": archivo_relativo,
            "hoja_origen": hoja,
            "celda_origen": f"{celda_excel(fila_fk_idx, col_fk)} / {celda_excel(fila_fi_idx, col_fi)}",
            "codigo_origen": "IF41/IF111",
            "descripcion_origen": descripcion_origen,
            "metodo_busqueda": "calculo_fk_fi",
            "fila_origen_excel": f"{fila_fk_idx + 1}/{fila_fi_idx + 1}",
            "columna_banco_excel": banco_fk["columna_excel"],
        })

    return registros


def resolver_y_construir_indicador_financiero(
    df,
    indicador,
    config,
    archivo_relativo,
    hoja,
    periodo,
    fecha_corte,
    incluir_agregados=False,
):
    """
    Resuelve un indicador financiero y devuelve registros + eventos.

    Para solvencia_proxy aplica la lógica especial:
    IF112/nombre directo o cálculo FK/FI.
    Para el resto de indicadores conserva la lógica original.
    """
    registros = []
    eventos = []

    if indicador == "solvencia_proxy":
        resultado, evs = resolver_solvencia_proxy_en_hoja(df, config, incluir_agregados)
        eventos.extend(evs)

        if resultado is None:
            return registros, eventos, None

        if resultado["tipo"] == "directo":
            candidato = resultado["candidato"]
            registros.extend(construir_registros_desde_candidato(
                df=df,
                candidato=candidato,
                indicador=indicador,
                config=config,
                archivo_relativo=archivo_relativo,
                hoja=hoja,
                periodo=periodo,
                fecha_corte=fecha_corte,
                incluir_agregados=incluir_agregados,
            ))
            return registros, eventos, candidato

        if resultado["tipo"] == "componentes_fk_fi":
            registros.extend(construir_registros_solvencia_proxy_desde_componentes(
                df=df,
                candidato_fk=resultado["candidato_fk"],
                candidato_fi=resultado["candidato_fi"],
                config=config,
                archivo_relativo=archivo_relativo,
                hoja=hoja,
                periodo=periodo,
                fecha_corte=fecha_corte,
                incluir_agregados=incluir_agregados,
            ))
            # Se retorna candidato_fk como referencia para auditoría de celda.
            return registros, eventos, resultado["candidato_fk"]

    candidato, evs = resolver_indicador_financiero_en_hoja(
        df=df,
        indicador=indicador,
        config=config,
        incluir_agregados=incluir_agregados,
    )
    eventos.extend(evs)

    if candidato:
        registros.extend(construir_registros_desde_candidato(
            df=df,
            candidato=candidato,
            indicador=indicador,
            config=config,
            archivo_relativo=archivo_relativo,
            hoja=hoja,
            periodo=periodo,
            fecha_corte=fecha_corte,
            incluir_agregados=incluir_agregados,
        ))

    return registros, eventos, candidato



## 14. Procesar un archivo Excel

Esta función está optimizada para velocidad:

- abre el Excel una sola vez;
- lee Balance una sola vez;
- lee Indicadores una sola vez;
- solo escanea hojas extra si falta algún indicador;
- se ejecuta en paralelo para todos los archivos.

In [15]:
def procesar_archivo_financiero(archivo: Path):
    """Procesa un archivo Excel y devuelve registros, eventos y auditoría."""
    archivo_relativo = str(archivo.relative_to(RAIZ_PROYECTO))
    registros = []
    eventos = []

    auditoria = {
        "archivo_origen": archivo_relativo,
        "archivo_nombre": archivo.name,
        "extension": archivo.suffix.lower(),
        "periodo": None,
        "fecha_corte": None,
        "celda_fecha": None,
        "valor_fecha_original": None,
        "hoja_balance": None,
        "metodo_balance": None,
        "hojas_indicadores": None,
        "estado": "iniciado",
        "detalle": None,
    }

    try:
        excel = pd.ExcelFile(archivo, engine=engine_excel(archivo))
        hojas = excel.sheet_names

        # Resolver Balance es obligatorio porque de ahí sale la fecha del periodo.
        hoja_balance, metodo_balance = resolver_hoja_balance(excel)
        auditoria["hoja_balance"] = hoja_balance
        auditoria["metodo_balance"] = metodo_balance

        if hoja_balance is None:
            auditoria["estado"] = "sin_balance"
            auditoria["detalle"] = "No se encontró hoja Balance ni cabecera ESTADO DE SITUACIÓN."
            eventos.append({
                "archivo_origen": archivo_relativo,
                "periodo": None,
                "indicador": None,
                "estado": "sin_balance",
                "detalle": auditoria["detalle"],
                "hoja": None,
                "celda": None,
            })
            return registros, eventos, auditoria

        # Lectura completa de Balance una sola vez.
        df_balance = pd.read_excel(excel, sheet_name=hoja_balance, header=None, dtype=object)

        fecha_info = detectar_fecha_en_balance(df_balance)
        auditoria.update(fecha_info)

        periodo = fecha_info["periodo"]
        fecha_corte = fecha_info["fecha_corte"]

        if periodo is None:
            eventos.append({
                "archivo_origen": archivo_relativo,
                "periodo": None,
                "indicador": None,
                "estado": "periodo_no_detectado_en_balance",
                "detalle": "No se pudo extraer fecha desde la cabecera de Balance.",
                "hoja": hoja_balance,
                "celda": None,
            })

        # Extraer indicadores contables desde Balance.
        for indicador, config in INDICADORES_BALANCE.items():
            candidato, evs = resolver_indicador_balance_en_hoja(df_balance, indicador, config, INCLUIR_AGREGADOS)

            for ev in evs:
                eventos.append({
                    "archivo_origen": archivo_relativo,
                    "periodo": periodo,
                    "indicador": ev.get("indicador"),
                    "estado": ev.get("estado"),
                    "detalle": ev.get("detalle"),
                    "hoja": hoja_balance,
                    "celda": candidato.get("celda") if candidato else None,
                    "candidatos": ev.get("candidatos"),
                })

            if candidato:
                registros.extend(construir_registros_desde_candidato(
                    df=df_balance,
                    candidato=candidato,
                    indicador=indicador,
                    config=config,
                    archivo_relativo=archivo_relativo,
                    hoja=hoja_balance,
                    periodo=periodo,
                    fecha_corte=fecha_corte,
                    incluir_agregados=INCLUIR_AGREGADOS,
                ))

        # Resolver hojas candidatas de indicadores financieros.
        hojas_indicadores = resolver_hojas_indicadores(excel)
        auditoria["hojas_indicadores"] = ", ".join([h for h, _ in hojas_indicadores]) if hojas_indicadores else None

        # Caché para no leer dos veces la misma hoja.
        cache_hojas = {hoja_balance: df_balance}

        indicadores_pendientes = set(INDICADORES_FINANCIEROS.keys())

        # Primero se prueban las hojas candidatas principales.
        for hoja_indicadores, metodo_hoja in hojas_indicadores:
            if not indicadores_pendientes:
                break

            if hoja_indicadores not in cache_hojas:
                cache_hojas[hoja_indicadores] = pd.read_excel(excel, sheet_name=hoja_indicadores, header=None, dtype=object)

            df_ind = cache_hojas[hoja_indicadores]

            for indicador in list(indicadores_pendientes):
                config = INDICADORES_FINANCIEROS[indicador]

                registros_indicador, evs, candidato = resolver_y_construir_indicador_financiero(
                    df=df_ind,
                    indicador=indicador,
                    config=config,
                    archivo_relativo=archivo_relativo,
                    hoja=hoja_indicadores,
                    periodo=periodo,
                    fecha_corte=fecha_corte,
                    incluir_agregados=INCLUIR_AGREGADOS,
                )

                for ev in evs:
                    eventos.append({
                        "archivo_origen": archivo_relativo,
                        "periodo": periodo,
                        "indicador": ev.get("indicador"),
                        "estado": ev.get("estado"),
                        "detalle": ev.get("detalle"),
                        "hoja": hoja_indicadores,
                        "celda": candidato.get("celda") if candidato else None,
                        "candidatos": ev.get("candidatos") or ev.get("candidatos_codigo") or ev.get("candidatos_nombre"),
                    })

                if candidato:
                    registros.extend(registros_indicador)
                    indicadores_pendientes.remove(indicador)

        # Si todavía faltan indicadores, se escanean hojas adicionales solo para esos faltantes.
        if ESCANEAR_HOJAS_EXTRA_SI_FALLA and indicadores_pendientes:
            hojas_ya_probadas = {h for h, _ in hojas_indicadores}
            hojas_extra = [h for h in hojas if h not in hojas_ya_probadas and h != hoja_balance]

            for hoja_extra in hojas_extra:
                if not indicadores_pendientes:
                    break

                try:
                    if hoja_extra not in cache_hojas:
                        cache_hojas[hoja_extra] = pd.read_excel(excel, sheet_name=hoja_extra, header=None, dtype=object)
                    df_extra = cache_hojas[hoja_extra]
                except Exception as e:
                    eventos.append({
                        "archivo_origen": archivo_relativo,
                        "periodo": periodo,
                        "indicador": None,
                        "estado": "error_lectura_hoja_extra",
                        "detalle": str(e),
                        "hoja": hoja_extra,
                        "celda": None,
                    })
                    continue

                for indicador in list(indicadores_pendientes):
                    config = INDICADORES_FINANCIEROS[indicador]

                    registros_indicador, evs, candidato = resolver_y_construir_indicador_financiero(
                        df=df_extra,
                        indicador=indicador,
                        config=config,
                        archivo_relativo=archivo_relativo,
                        hoja=hoja_extra,
                        periodo=periodo,
                        fecha_corte=fecha_corte,
                        incluir_agregados=INCLUIR_AGREGADOS,
                    )

                    # Solo registramos eventos de hoja extra cuando realmente encuentra algo.
                    if candidato:
                        eventos.append({
                            "archivo_origen": archivo_relativo,
                            "periodo": periodo,
                            "indicador": indicador,
                            "estado": "indicador_encontrado_en_hoja_extra",
                            "detalle": "El indicador no estaba en la hoja principal de indicadores y se encontró en una hoja extra.",
                            "hoja": hoja_extra,
                            "celda": candidato.get("celda"),
                        })

                        for ev in evs:
                            eventos.append({
                                "archivo_origen": archivo_relativo,
                                "periodo": periodo,
                                "indicador": ev.get("indicador"),
                                "estado": ev.get("estado"),
                                "detalle": ev.get("detalle"),
                                "hoja": hoja_extra,
                                "celda": candidato.get("celda") if candidato else None,
                                "candidatos": ev.get("candidatos") or ev.get("candidatos_codigo") or ev.get("candidatos_nombre"),
                            })

                        registros.extend(registros_indicador)
                        indicadores_pendientes.remove(indicador)

        # Registrar indicadores que definitivamente no se encontraron.
        for indicador in indicadores_pendientes:
            eventos.append({
                "archivo_origen": archivo_relativo,
                "periodo": periodo,
                "indicador": indicador,
                "estado": "indicador_financiero_no_encontrado_final",
                "detalle": "No se encontró por código ni por nombre/variantes en las hojas revisadas.",
                "hoja": auditoria.get("hojas_indicadores"),
                "celda": None,
            })

        auditoria["estado"] = "ok"
        auditoria["detalle"] = f"Registros extraídos: {len(registros)}"
        return registros, eventos, auditoria

    except Exception as e:
        auditoria["estado"] = "error_archivo"
        auditoria["detalle"] = str(e)
        eventos.append({
            "archivo_origen": archivo_relativo,
            "periodo": None,
            "indicador": None,
            "estado": "error_archivo",
            "detalle": str(e),
            "hoja": None,
            "celda": None,
        })
        return registros, eventos, auditoria

## 15. Listar archivos Excel

Se leen todos los archivos dentro de `00_datos_crudos`, incluyendo subcarpetas.

Se ignoran archivos temporales de Excel que empiezan con `~$`.

In [16]:
archivos_excel = sorted([
    p for p in RAW_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in EXTENSIONES_EXCEL
    and not p.name.startswith("~$")
])

print(f"Total de archivos Excel encontrados: {len(archivos_excel)}")

for i, archivo in enumerate(archivos_excel[:20], start=1):
    print(f"{i:03d}. {archivo.relative_to(RAIZ_PROYECTO)}")

if len(archivos_excel) > 20:
    print(f"... y {len(archivos_excel) - 20} archivos más")

Total de archivos Excel encontrados: 205
001. 00_datos_crudos\BOL BANCOS PRIVADOS 28022015 act.xlsx
002. 00_datos_crudos\BOL BANCOS PRIVADOS 30042015 act.xlsx
003. 00_datos_crudos\BOL BANCOS PRIVADOS 30062015 act.xlsx
004. 00_datos_crudos\BOL BANCOS PRIVADOS 30072015 act.xlsx
005. 00_datos_crudos\BOL BANCOS PRIVADOS 30092014.xlsx
006. 00_datos_crudos\BOL BANCOS PRIVADOS 30112014.xlsx
007. 00_datos_crudos\BOL BANCOS PRIVADOS 31012015 act.xlsx
008. 00_datos_crudos\BOL BANCOS PRIVADOS 31032015 act.xlsx
009. 00_datos_crudos\BOL BANCOS PRIVADOS 31052015 act.xlsx
010. 00_datos_crudos\BOL BANCOS PRIVADOS 31082014.xlsm
011. 00_datos_crudos\BOL BANCOS PRIVADOS 31102014.xlsx
012. 00_datos_crudos\BOL BANCOS PRIVADOS 31122014.xlsx
013. 00_datos_crudos\BOL BCOS PRIVADOS 201508.xlsx
014. 00_datos_crudos\BOL BCOS PRIVADOS 201509.xlsx
015. 00_datos_crudos\BOL BCOS PRIVADOS 201510.xlsx
016. 00_datos_crudos\BOL BCOS PRIVADOS 201511.xlsx
017. 00_datos_crudos\BOL BCOS PRIVADOS 201512.xlsx
018. 00_datos_cr

## 16. Procesar todos los archivos

Esta celda ejecuta la extracción completa.

Usa paralelización con `ThreadPoolExecutor`, pero mantiene cada archivo aislado para evitar mezclar resultados.

In [17]:
resultados = []
eventos = []
auditoria = []

if not archivos_excel:
    print("No se encontraron archivos Excel para procesar.")
else:
    print(f"Procesando {len(archivos_excel)} archivos con N_WORKERS={N_WORKERS}...")

    if N_WORKERS == 1:
        # Modo secuencial: útil para depurar errores archivo por archivo.
        for i, archivo in enumerate(archivos_excel, start=1):
            print(f"[{i:03d}/{len(archivos_excel):03d}] {archivo.name}")
            regs, evs, aud = procesar_archivo_financiero(archivo)
            resultados.extend(regs)
            eventos.extend(evs)
            auditoria.append(aud)
    else:
        # Modo paralelo: acelera la lectura cuando existen muchos archivos.
        with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
            futuros = {executor.submit(procesar_archivo_financiero, archivo): archivo for archivo in archivos_excel}

            for i, futuro in enumerate(as_completed(futuros), start=1):
                archivo = futuros[futuro]
                try:
                    regs, evs, aud = futuro.result()
                except Exception as e:
                    regs = []
                    evs = [{
                        "archivo_origen": str(archivo.relative_to(RAIZ_PROYECTO)),
                        "periodo": None,
                        "indicador": None,
                        "estado": "error_futuro",
                        "detalle": str(e),
                        "hoja": None,
                        "celda": None,
                    }]
                    aud = {
                        "archivo_origen": str(archivo.relative_to(RAIZ_PROYECTO)),
                        "archivo_nombre": archivo.name,
                        "estado": "error_futuro",
                        "detalle": str(e),
                    }

                resultados.extend(regs)
                eventos.extend(evs)
                auditoria.append(aud)
                print(f"[{i:03d}/{len(archivos_excel):03d}] OK: {archivo.name}")


df_dataset_largo = pd.DataFrame(resultados)
df_eventos = pd.DataFrame(eventos)
df_auditoria_archivos = pd.DataFrame(auditoria)

# Aseguramos que el dataset tenga siempre las columnas finales, aunque algún archivo falle.
for col in COLUMNAS_DATASET_FINAL:
    if col not in df_dataset_largo.columns:
        df_dataset_largo[col] = pd.Series(dtype="object")

df_dataset_largo = df_dataset_largo[COLUMNAS_DATASET_FINAL]

print("\nProceso terminado")
print("Filas extraídas:", len(df_dataset_largo))
print("Eventos registrados:", len(df_eventos))
print("Archivos auditados:", len(df_auditoria_archivos))

Procesando 205 archivos con N_WORKERS=6...
[001/205] OK: BOL BANCOS PRIVADOS 28022015 act.xlsx
[002/205] OK: BOL BANCOS PRIVADOS 30042015 act.xlsx
[003/205] OK: BOL BANCOS PRIVADOS 30112014.xlsx
[004/205] OK: BOL BANCOS PRIVADOS 30072015 act.xlsx
[005/205] OK: BOL BANCOS PRIVADOS 30092014.xlsx
[006/205] OK: BOL BANCOS PRIVADOS 30062015 act.xlsx
[007/205] OK: BOL BANCOS PRIVADOS 31032015 act.xlsx
[008/205] OK: BOL BANCOS PRIVADOS 31012015 act.xlsx
[009/205] OK: BOL BANCOS PRIVADOS 31052015 act.xlsx
[010/205] OK: BOL BANCOS PRIVADOS 31082014.xlsm
[011/205] OK: BOL BANCOS PRIVADOS 31102014.xlsx
[012/205] OK: BOL BANCOS PRIVADOS 31122014.xlsx
[013/205] OK: BOL BCOS PRIVADOS 201508.xlsx
[014/205] OK: BOL BCOS PRIVADOS 201509.xlsx
[015/205] OK: BOL BCOS PRIVADOS 201510.xlsx
[016/205] OK: BOL BCOS PRIVADOS 201511.xlsx
[017/205] OK: BOL BCOS PRIVADOS 201512.xlsx
[018/205] OK: BOL200903BCOS.xls
[019/205] OK: BOL201102B PLANTILLA web .xls
[020/205] OK: BOL200907BCOS.xls
[021/205] OK: BOL-FIN-BCO

## 17. Control visual del dataset largo

Esta vista permite confirmar rápidamente si los indicadores están saliendo con periodo, banco, valor y celda de origen.

In [18]:
display(df_dataset_largo.head(100))

,periodo,fecha_corte,banco_original,banco_estandarizado,indicador,valor,unidad,sentido,archivo_origen,hoja_origen,celda_origen,codigo_origen,descripcion_origen,metodo_busqueda,fila_origen_excel,columna_banco_excel
0,2015-02,2015-02-28,BP GUAYAQUIL,GUAYAQUIL,activos_totales,3.956442e+06,miles_usd,mayor_es_tamano,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,BALANCE,D621,NaN,TOTAL ACTIVO,etiqueta_balance,621,D
1,2015-02,2015-02-28,BP PACIFICO,PACIFICO,activos_totales,4.470343e+06,miles_usd,mayor_es_tamano,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,BALANCE,E621,NaN,TOTAL ACTIVO,etiqueta_balance,621,E
2,2015-02,2015-02-28,BP PICHINCHA,PICHINCHA,activos_totales,9.510421e+06,miles_usd,mayor_es_tamano,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,BALANCE,F621,NaN,TOTAL ACTIVO,etiqueta_balance,621,F
3,2015-02,2015-02-28,BP PRODUBANCO,PRODUBANCO,activos_totales,3.737498e+06,miles_usd,mayor_es_tamano,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,BALANCE,G621,NaN,TOTAL ACTIVO,etiqueta_balance,621,G
4,2015-02,2015-02-28,BP AUSTRO,AUSTRO,activos_totales,1.533289e+06,miles_usd,mayor_es_tamano,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,BALANCE,I621,NaN,TOTAL ACTIVO,etiqueta_balance,621,I
5,2015-02,2015-02-28,BP BOLIVARIANO,BOLIVARIANO,activos_totales,2.748532e+06,miles_usd,mayor_es_tamano,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,BALANCE,J621,NaN,TOTAL ACTIVO,etiqueta_balance,621,J
6,2015-02,2015-02-28,BP SOLIDARIO,SOLIDARIO,activos_totales,7.315736e+05,miles_usd,mayor_es_tamano,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,BALANCE,K621,NaN,TOTAL ACTIVO,etiqueta_balance,621,K
7,2015-02,2015-02-28,BP GENERAL RUMIÑAHUI,GENERAL RUMIÑAHUI,activos_totales,6.310042e+05,miles_usd,mayor_es_tamano,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,BALANCE,L621,NaN,TOTAL ACTIVO,etiqueta_balance,621,L
8,2015-02,2015-02-28,BP INTERNACIONAL,INTERNACIONAL,activos_totales,2.598928e+06,miles_usd,mayor_es_tamano,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,BALANCE,M621,NaN,TOTAL ACTIVO,etiqueta_balance,621,M
9,2015-02,2015-02-28,BP MACHALA,MACHALA,activos_totales,6.901549e+05,miles_usd,mayor_es_tamano,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,BALANCE,N621,NaN,TOTAL ACTIVO,etiqueta_balance,621,N


## 18. Matriz de control por archivo e indicador

Esta matriz muestra cuántos bancos se extrajeron por archivo, periodo e indicador.

Si un indicador tiene cero bancos en algún archivo, revisa `df_eventos`.

In [19]:
if df_dataset_largo.empty:
    print("No hay datos para construir matriz de control.")
    matriz_control = pd.DataFrame()
else:
    matriz_control = (
        df_dataset_largo
        .pivot_table(
            index=["archivo_origen", "periodo"],
            columns="indicador",
            values="banco_estandarizado",
            aggfunc=pd.Series.nunique,
            fill_value=0,
        )
        .reset_index()
    )

    columnas_indicadores_esperadas = list(INDICADORES_BALANCE.keys()) + list(INDICADORES_FINANCIEROS.keys())

    for col in columnas_indicadores_esperadas:
        if col not in matriz_control.columns:
            matriz_control[col] = 0

    matriz_control = matriz_control[["archivo_origen", "periodo"] + columnas_indicadores_esperadas]
    display(matriz_control.head(160))

indicador,archivo_origen,periodo,activos_totales,pasivos_totales,patrimonio,roe,morosidad,solvencia_proxy
0,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,2015-02,23,23,23,23,23,23
1,00_datos_crudos\BOL BANCOS PRIVADOS 30042015 a...,2015-04,23,23,23,23,23,23
2,00_datos_crudos\BOL BANCOS PRIVADOS 30062015 a...,2015-06,23,23,23,23,23,23
3,00_datos_crudos\BOL BANCOS PRIVADOS 30072015 a...,2015-07,23,23,23,23,23,23
4,00_datos_crudos\BOL BANCOS PRIVADOS 30092014.xlsx,2014-09,27,27,27,27,27,27
5,00_datos_crudos\BOL BANCOS PRIVADOS 30112014.xlsx,2014-11,27,27,27,27,27,27
6,00_datos_crudos\BOL BANCOS PRIVADOS 31012015 a...,2015-01,23,23,23,23,23,23
7,00_datos_crudos\BOL BANCOS PRIVADOS 31032015 a...,2015-03,23,23,23,23,23,23
8,00_datos_crudos\BOL BANCOS PRIVADOS 31052015 a...,2015-05,23,23,23,23,23,23
9,00_datos_crudos\BOL BANCOS PRIVADOS 31082014.xlsm,2014-08,27,27,27,27,27,27


## 19. Revisión de valores nulos

Recuerda la interpretación:

- `NaN` significa que el banco existía en ese periodo porque apareció en el encabezado, pero el valor no fue reportado o venía con error de Excel.
- Si un banco no existía en ese periodo, no aparece fila en el dataset.
- `0` es un valor real y no debe confundirse con `NaN`.

In [20]:
if df_dataset_largo.empty:
    print("No hay datos para revisar nulos.")
else:
    resumen_nulos = (
        df_dataset_largo
        .groupby("indicador", dropna=False)
        .agg(
            registros=("valor", "size"),
            nulos=("valor", lambda s: int(s.isna().sum())),
            bancos=("banco_estandarizado", "nunique"),
            archivos=("archivo_origen", "nunique"),
        )
        .reset_index()
        .sort_values("indicador")
    )

    display(resumen_nulos)

    df_nulos = df_dataset_largo[df_dataset_largo["valor"].isna()].copy()
    print("Total de valores NaN:", len(df_nulos))

    if not df_nulos.empty:
        display(df_nulos[[
            "archivo_origen", "periodo", "banco_original", "banco_estandarizado",
            "indicador", "valor", "hoja_origen", "celda_origen", "metodo_busqueda",
        ]].head(200))

,indicador,registros,nulos,bancos,archivos
0,activos_totales,4963,49,32,205
1,morosidad,4963,5,32,205
2,pasivos_totales,4963,49,32,205
3,patrimonio,4963,49,32,205
4,roe,4963,32,32,205
5,solvencia_proxy,4915,50,32,203


Total de valores NaN: 234


,archivo_origen,periodo,banco_original,banco_estandarizado,indicador,valor,hoja_origen,celda_origen,metodo_busqueda
293,00_datos_crudos\BOL BANCOS PRIVADOS 30112014.xlsx,2014-11,BP SUDAMERICANO,SUDAMERICANO,activos_totales,NaN,BALANCE,W621,etiqueta_balance
294,00_datos_crudos\BOL BANCOS PRIVADOS 30112014.xlsx,2014-11,BP TERRITORIAL,TERRITORIAL,activos_totales,NaN,BALANCE,X621,etiqueta_balance
295,00_datos_crudos\BOL BANCOS PRIVADOS 30112014.xlsx,2014-11,BP UNIBANCO,UNIBANCO,activos_totales,NaN,BALANCE,Y621,etiqueta_balance
320,00_datos_crudos\BOL BANCOS PRIVADOS 30112014.xlsx,2014-11,BP SUDAMERICANO,SUDAMERICANO,pasivos_totales,NaN,BALANCE,W809,etiqueta_balance
321,00_datos_crudos\BOL BANCOS PRIVADOS 30112014.xlsx,2014-11,BP TERRITORIAL,TERRITORIAL,pasivos_totales,NaN,BALANCE,X809,etiqueta_balance
...,...,...,...,...,...,...,...,...,...
28705,00_datos_crudos\NUEVA PLANTILLA BCOS PRIVADOS ...,2014-04,BP UNIBANCO,UNIBANCO,activos_totales,NaN,BALANCE,Y621,etiqueta_balance
28731,00_datos_crudos\NUEVA PLANTILLA BCOS PRIVADOS ...,2014-04,BP TERRITORIAL,TERRITORIAL,pasivos_totales,NaN,BALANCE,X809,etiqueta_balance
28732,00_datos_crudos\NUEVA PLANTILLA BCOS PRIVADOS ...,2014-04,BP UNIBANCO,UNIBANCO,pasivos_totales,NaN,BALANCE,Y809,etiqueta_balance
28758,00_datos_crudos\NUEVA PLANTILLA BCOS PRIVADOS ...,2014-04,BP TERRITORIAL,TERRITORIAL,patrimonio,NaN,BALANCE,X854,etiqueta_balance


## 20. Revisión de eventos

Los eventos no siempre son errores. Algunos eventos son auditorías útiles, por ejemplo:

- `fallback_nombre_usado`: no se encontró código útil y se usó nombre.
- `duplicado_codigo_resuelto`: se encontraron varias filas por código y se escogió la fila con más valores.
- `indicador_encontrado_en_hoja_extra`: el indicador apareció fuera de la hoja principal esperada.

In [21]:
if df_eventos.empty:
    print("No se registraron eventos. Excelente.")
else:
    display(df_eventos.head(200))

    print("Resumen de eventos:")
    display(
        df_eventos
        .groupby(["estado", "indicador"], dropna=False)
        .size()
        .reset_index(name="conteo")
        .sort_values(["estado", "indicador"], na_position="last")
    )

,archivo_origen,periodo,indicador,estado,detalle,hoja,celda,candidatos
0,00_datos_crudos\BOL200903BCOS.xls,2009-03,patrimonio,duplicado_balance_resuelto,Se encontraron 2 coincidencias; se usó la fila...,BAL SAB II,C1221,"[(C1221, 25, 25), (C649, 25, 25)]"
1,00_datos_crudos\BOL200903BCOS.xls,2009-03,solvencia_proxy,duplicado_codigo_resuelto,Se encontraron 2 coincidencias por código IF11...,INDIC,B81,"[(B81, 25, 25), (B80, 25, 25)]"
2,00_datos_crudos\BOL200903BCOS.xls,2009-03,morosidad,fallback_nombre_usado,No se encontró una fila útil por código; se us...,INDIC,C24,"[(C24, 25, 25)]"
3,00_datos_crudos\BOL201102B PLANTILLA web .xls,2011-02,patrimonio,duplicado_balance_resuelto,Se encontraron 2 coincidencias; se usó la fila...,BAL SAB II,C1225,"[(C1225, 25, 25), (C650, 25, 25)]"
4,00_datos_crudos\BOL201102B PLANTILLA web .xls,2011-02,solvencia_proxy,duplicado_codigo_resuelto,Se encontraron 2 coincidencias por código IF11...,INDIC,B82,"[(B82, 25, 25), (B81, 25, 25)]"
...,...,...,...,...,...,...,...,...
195,00_datos_crudos\FINANCIERO MENSUAL BANCA PRIVA...,2023-02,solvencia_proxy,fallback_nombre_usado,No se encontró una fila útil por IF112; se usó...,INDICADORES,B88,"[(B88, 24, 24)]"
196,00_datos_crudos\FINANCIERO MENSUAL BANCA PRIVA...,2023-02,roe,fallback_nombre_usado,No se encontró una fila útil por código; se us...,INDICADORES,B53,"[(B53, 24, 24), (B47, 0, 0)]"
197,00_datos_crudos\FINANCIERO MENSUAL BANCA PRIVA...,2023-02,roe,duplicado_nombre_resuelto,Se encontraron 2 coincidencias por nombre; se ...,INDICADORES,B53,"[(B53, 24, 24), (B47, 0, 0)]"
198,00_datos_crudos\FINANCIERO MENSUAL BANCA PRIVA...,2023-02,morosidad,fallback_nombre_usado,No se encontró una fila útil por código; se us...,INDICADORES,B29,"[(B29, 24, 24)]"


Resumen de eventos:


,estado,indicador,conteo
0,componente_fk_no_encontrado,solvencia_proxy,2
1,duplicado_balance_resuelto,patrimonio,40
2,duplicado_codigo_resuelto,roe,32
3,duplicado_codigo_resuelto,solvencia_proxy,33
4,duplicado_nombre_resuelto,roe,52
5,fallback_fi_nombre_usado,solvencia_proxy,2
6,fallback_nombre_usado,morosidad,95
7,fallback_nombre_usado,roe,52
8,fallback_nombre_usado,solvencia_proxy,50
9,indicador_financiero_no_encontrado_final,solvencia_proxy,2


## 21. Auditoría de archivos procesados

Permite revisar qué hoja se usó como Balance, qué periodo se detectó y si hubo errores de lectura.

In [22]:
if df_auditoria_archivos.empty:
    print("No hay auditoría de archivos.")
else:
    columnas_auditoria = [
        "archivo_origen", "periodo", "fecha_corte", "celda_fecha", "valor_fecha_original",
        "hoja_balance", "metodo_balance", "hojas_indicadores", "estado", "detalle",
    ]
    columnas_auditoria = [c for c in columnas_auditoria if c in df_auditoria_archivos.columns]

    display(df_auditoria_archivos[columnas_auditoria].sort_values("archivo_origen").head(200))

    print("Archivos sin periodo detectado desde Balance:")
    if "periodo" in df_auditoria_archivos.columns:
        display(df_auditoria_archivos[df_auditoria_archivos["periodo"].isna()][columnas_auditoria])
    else:
        display(df_auditoria_archivos)

,archivo_origen,periodo,fecha_corte,celda_fecha,valor_fecha_original,hoja_balance,metodo_balance,hojas_indicadores,estado,detalle
0,00_datos_crudos\BOL BANCOS PRIVADOS 28022015 a...,2015-02,2015-02-28,B4,2015-02-28 00:00:00,BALANCE,nombre_balance,INDICADORES,ok,Registros extraídos: 138
1,00_datos_crudos\BOL BANCOS PRIVADOS 30042015 a...,2015-04,2015-04-30,B4,2015-04-30 00:00:00,BALANCE,nombre_balance,INDICADORES,ok,Registros extraídos: 138
5,00_datos_crudos\BOL BANCOS PRIVADOS 30062015 a...,2015-06,2015-06-30,A4,2015-06-30 00:00:00,BALANCE,nombre_balance,INDICADORES,ok,Registros extraídos: 138
3,00_datos_crudos\BOL BANCOS PRIVADOS 30072015 a...,2015-07,2015-07-31,A4,2015-07-31 00:00:00,BALANCE,nombre_balance,INDICADORES,ok,Registros extraídos: 138
4,00_datos_crudos\BOL BANCOS PRIVADOS 30092014.xlsx,2014-09,2014-09-30,B4,2014-09-30 00:00:00,BALANCE,nombre_balance,INDICADORES,ok,Registros extraídos: 162
...,...,...,...,...,...,...,...,...,...,...
196,00_datos_crudos\NUEVA PLANTILLA BCOS PRIVADOS ...,2014-01,2014-01-31,B4,2014-01-31 00:00:00,BALANCE,nombre_balance,INDICADORES,ok,Registros extraídos: 156
195,00_datos_crudos\NUEVA PLANTILLA BCOS PRIVADOS ...,2014-02,2014-02-28,B4,2014-02-28 00:00:00,BALANCE,nombre_balance,INDICADORES,ok,Registros extraídos: 156
197,00_datos_crudos\NUEVA PLANTILLA BCOS PRIVADOS ...,2014-03,2014-03-31,B4,2014-03-31 00:00:00,BALANCE,nombre_balance,INDICADORES,ok,Registros extraídos: 162
198,00_datos_crudos\NUEVA PLANTILLA BCOS PRIVADOS ...,2014-04,2014-04-30,B4,2014-04-30 00:00:00,BALANCE,nombre_balance,"INDICADORES, VALID INDICADORES, INDIC MAF",ok,Registros extraídos: 162


Archivos sin periodo detectado desde Balance:


,archivo_origen,periodo,fecha_corte,celda_fecha,valor_fecha_original,hoja_balance,metodo_balance,hojas_indicadores,estado,detalle


## 22. Resolver duplicados finales y construir `dataset_financiero_limpio`

Aunque la lógica ya selecciona una fila por indicador, esta celda elimina duplicados finales en caso de que algún archivo tenga estructuras especiales.

Criterio de prioridad:

1. valor no nulo;
2. búsqueda por código;
3. búsqueda por nombre;
4. búsqueda por etiqueta de Balance.

Después se genera el dataset final limpio solicitado con únicamente estas columnas:

```text
periodo | banco_estandarizado | indicador | valor | unidad | sentido
```

Este será el archivo principal para Streamlit y análisis financiero.

Nota adicional: para `roe`, `morosidad` y `solvencia_proxy`, si Excel almacena internamente una celda porcentual como decimal —por ejemplo, `8,95%` leído como `0.0895`— el dataset limpio conserva el porcentaje visible esperado como `8.95`, redondeado a 2 decimales.


In [23]:
def prioridad_metodo(metodo):
    """Asigna prioridad a cada método de búsqueda para resolver duplicados finales."""
    prioridades = {
        "codigo": 1,            # Mayor prioridad: código estándar del indicador.
        "etiqueta_balance": 2,  # Usado para activos, pasivos y patrimonio.
        "nombre": 3,            # Respaldo cuando el código no aparece o no tiene valores útiles.
        "calculo_fk_fi": 4,    # Respaldo específico para solvencia_proxy usando FK/FI.
    }
    return prioridades.get(metodo, 9)


# Conservamos una versión rica con trazabilidad para auditoría interna.
df_dataset_final = df_dataset_largo.copy()


# Reaplicamos la normalización bancaria antes de resolver duplicados.
# Esto permite corregir datasets ya extraídos con nombres antiguos sin volver a leer los 205 archivos,
# siempre que exista la columna banco_original.
if not df_dataset_final.empty and "banco_original" in df_dataset_final.columns:
    df_dataset_final["banco_estandarizado"] = df_dataset_final["banco_original"].apply(estandarizar_nombre_banco)

if not df_dataset_final.empty:
    # Convertimos valor a numérico. Los errores, vacíos o #DIV/0! quedan como NaN.
    df_dataset_final["valor"] = pd.to_numeric(df_dataset_final["valor"], errors="coerce")

    # Columna auxiliar: False para valores reales, True para valores nulos.
    # Al ordenar ascendente, los valores reales quedan primero.
    df_dataset_final["_valor_es_nulo"] = df_dataset_final["valor"].isna()

    # Columna auxiliar para preferir código sobre nombre cuando existan duplicados.
    df_dataset_final["_prioridad_metodo"] = df_dataset_final["metodo_busqueda"].map(prioridad_metodo)

    claves_auditoria = ["periodo", "banco_estandarizado", "indicador", "archivo_origen"]

    duplicados_antes = (
        df_dataset_final
        .groupby(claves_auditoria, dropna=False)
        .size()
        .reset_index(name="conteo")
        .query("conteo > 1")
    )

    print("Duplicados antes de depurar en dataset de auditoría:", len(duplicados_antes))

    # Se conserva el mejor registro por periodo, banco, indicador y archivo.
    df_dataset_final = (
        df_dataset_final
        .sort_values(
            claves_auditoria + ["_valor_es_nulo", "_prioridad_metodo"],
            ascending=[True, True, True, True, True, True],
            na_position="last",
        )
        .drop_duplicates(subset=claves_auditoria, keep="first")
        .drop(columns=["_valor_es_nulo", "_prioridad_metodo"])
        .sort_values(["periodo", "banco_estandarizado", "indicador", "archivo_origen"], na_position="last")
        .reset_index(drop=True)
    )

# =============================================================================
# Dataset final limpio solicitado
# =============================================================================
# Este dataset elimina columnas de trazabilidad y deja solo las variables necesarias
# para análisis, rankings y dashboard.
df_dataset_financiero_limpio = df_dataset_final.copy()

if not df_dataset_financiero_limpio.empty:
    df_dataset_financiero_limpio["valor"] = pd.to_numeric(df_dataset_financiero_limpio["valor"], errors="coerce")
    df_dataset_financiero_limpio["_valor_es_nulo"] = df_dataset_financiero_limpio["valor"].isna()
    df_dataset_financiero_limpio["_prioridad_metodo"] = df_dataset_financiero_limpio["metodo_busqueda"].map(prioridad_metodo)

    claves_limpias = ["periodo", "banco_estandarizado", "indicador"]

    duplicados_limpio_antes = (
        df_dataset_financiero_limpio
        .groupby(claves_limpias, dropna=False)
        .size()
        .reset_index(name="conteo")
        .query("conteo > 1")
    )

    print("Duplicados antes de depurar en dataset limpio:", len(duplicados_limpio_antes))

    # Si existen duplicados entre archivos del mismo periodo, se prioriza el valor no nulo
    # y luego el método más confiable.
    df_dataset_financiero_limpio = (
        df_dataset_financiero_limpio
        .sort_values(
            claves_limpias + ["_valor_es_nulo", "_prioridad_metodo", "archivo_origen"],
            ascending=[True, True, True, True, True, True],
            na_position="last",
        )
        .drop_duplicates(subset=claves_limpias, keep="first")
        .drop(columns=["_valor_es_nulo", "_prioridad_metodo"])
        [COLUMNAS_DATASET_LIMPIO]
        .sort_values(["periodo", "banco_estandarizado", "indicador"], na_position="last")
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Normalización final de escala para el dataset limpio del dashboard
    # -------------------------------------------------------------------------
    # El boletín oficial reporta las cuentas monetarias en miles de dólares.
    # Para que el dashboard trabaje con una escala más legible, únicamente en
    # dataset_financiero_limpio convertimos esos valores a millones de dólares.
    #
    # Ejemplo:
    # 640619.09308 miles_usd / 1000 = 640.62 millones_usd
    #
    # El dataset de auditoría df_dataset_final se mantiene con los valores
    # originales y trazabilidad de hoja/celda/archivo.
    indicadores_monetarios = ["activos_totales", "pasivos_totales", "patrimonio"]
    indicadores_porcentaje = ["roe", "morosidad", "solvencia_proxy"]

    df_dataset_financiero_limpio["unidad"] = (
        df_dataset_financiero_limpio["unidad"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # -------------------------------------------------------------------------
    # Corrección de escala para porcentajes leídos desde Excel
    # -------------------------------------------------------------------------
    # Excel puede mostrar 8,95% pero almacenar internamente 0.0895.
    # Si pandas lee 0.0895, el dataset no debe quedarse con 0.09,
    # sino con 8.95. Esta corrección aplica únicamente a:
    # roe, morosidad y solvencia_proxy.
    mascara_porcentajes_excel_decimal = (
        df_dataset_financiero_limpio["indicador"].isin(indicadores_porcentaje)
        & df_dataset_financiero_limpio["valor"].notna()
        & (df_dataset_financiero_limpio["valor"] != 0)
        & (df_dataset_financiero_limpio["valor"].abs() <= 1)
    )

    df_dataset_financiero_limpio.loc[mascara_porcentajes_excel_decimal, "valor"] = (
        df_dataset_financiero_limpio.loc[mascara_porcentajes_excel_decimal, "valor"] * 100
    )

    mascara_monetarios = (
        df_dataset_financiero_limpio["indicador"].isin(indicadores_monetarios)
        | (df_dataset_financiero_limpio["unidad"] == "miles_usd")
    )

    df_dataset_financiero_limpio.loc[mascara_monetarios, "valor"] = (
        df_dataset_financiero_limpio.loc[mascara_monetarios, "valor"] / 1000
    )

    df_dataset_financiero_limpio.loc[mascara_monetarios, "unidad"] = "millones_usd"

    mascara_porcentajes = (
        df_dataset_financiero_limpio["indicador"].isin(indicadores_porcentaje)
        | (df_dataset_financiero_limpio["unidad"] == "porcentaje")
    )

    # Los valores monetarios y porcentuales se redondean a 2 decimales para que
    # el dataset final sea consistente y esté listo para el dashboard.
    df_dataset_financiero_limpio.loc[mascara_monetarios | mascara_porcentajes, "valor"] = (
        df_dataset_financiero_limpio.loc[mascara_monetarios | mascara_porcentajes, "valor"]
        .round(2)
    )
else:
    df_dataset_financiero_limpio = pd.DataFrame(columns=COLUMNAS_DATASET_LIMPIO)

print("Filas dataset de auditoría:", len(df_dataset_final))
print("Filas dataset_financiero_limpio:", len(df_dataset_financiero_limpio))
display(df_dataset_financiero_limpio.head(120))

Duplicados antes de depurar en dataset de auditoría: 0
Duplicados antes de depurar en dataset limpio: 0
Filas dataset de auditoría: 29730
Filas dataset_financiero_limpio: 29730


,periodo,banco_estandarizado,indicador,valor,unidad,sentido
0,2009-01,AMAZONAS,activos_totales,111.57,millones_usd,mayor_es_tamano
1,2009-01,AMAZONAS,morosidad,4.55,porcentaje,menor_es_mejor
2,2009-01,AMAZONAS,pasivos_totales,100.08,millones_usd,informativo
3,2009-01,AMAZONAS,patrimonio,11.38,millones_usd,mayor_es_mejor
4,2009-01,AMAZONAS,roe,11.41,porcentaje,mayor_es_mejor
5,2009-01,AMAZONAS,solvencia_proxy,8.65,porcentaje,mayor_es_mejor
6,2009-01,AUSTRO,activos_totales,646.02,millones_usd,mayor_es_tamano
7,2009-01,AUSTRO,morosidad,6.10,porcentaje,menor_es_mejor
8,2009-01,AUSTRO,pasivos_totales,588.88,millones_usd,informativo
9,2009-01,AUSTRO,patrimonio,56.66,millones_usd,mayor_es_mejor


## 23. Construir dataset de bancos únicos

Este dataset sirve como fuente para filtros del dashboard.

Contiene nombres sin repetir, periodos donde aparece cada banco y cuántos registros tiene.

In [24]:
if df_dataset_final.empty:
    df_bancos = pd.DataFrame(columns=[
        "banco_estandarizado", "nombres_originales", "primer_periodo", "ultimo_periodo",
        "numero_periodos", "numero_archivos", "numero_registros",
    ])
else:
    df_bancos = (
        df_dataset_final
        .dropna(subset=["banco_estandarizado"])
        .groupby("banco_estandarizado", dropna=False)
        .agg(
            nombres_originales=("banco_original", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
            primer_periodo=("periodo", "min"),
            ultimo_periodo=("periodo", "max"),
            numero_periodos=("periodo", "nunique"),
            numero_archivos=("archivo_origen", "nunique"),
            numero_registros=("indicador", "size"),
        )
        .reset_index()
        .sort_values("banco_estandarizado")
    )

print("Bancos únicos encontrados:", len(df_bancos))
display(df_bancos.head(100))

Bancos únicos encontrados: 32


,banco_estandarizado,nombres_originales,primer_periodo,ultimo_periodo,numero_periodos,numero_archivos,numero_registros
0,AMAZONAS,BP AMAZONAS,2009-01,2026-03,205,205,1228
1,ATLANTIDA,BANCO ATLÁNTIDA S.A.,2025-06,2026-03,10,10,60
2,AUSTRO,BP AUSTRO,2009-01,2026-03,205,205,1228
3,BOLIVARIANO,BP BOLIVARIANO,2009-01,2026-03,205,205,1228
4,CAPITAL,BP CAPITAL,2009-01,2026-03,205,205,1228
5,CITIBANK,BP CITIBANK,2009-01,2026-03,205,205,1228
6,COFIEC,BP COFIEC,2009-01,2015-07,79,79,474
7,COMERCIAL DE MANABI,BP BANCO COMERCIAL DE MANABI | BP COMERCIAL DE...,2009-01,2026-03,205,205,1228
8,COOPNACIONAL,BP COOPNACIONAL,2011-09,2026-03,173,173,1036
9,D-MIRO,BP D MIRO S.A. | BP D-MIRO | BP D-MIRO S.A.,2011-08,2025-05,164,164,982


## 23.1 Validación de bancos normalizados

Esta validación confirma que las variantes reportadas quedaron consolidadas en un solo nombre estándar.

In [25]:
# Validación específica de los bancos que presentaban duplicidad por nombres históricos.
patrones_revision = ["DINERS", "D-MIRO", "D MIRO", "FINCA", "VISIONFUND", "DESARROLLO", "CODESARROLLO"]

if "df_dataset_final" in globals() and not df_dataset_final.empty:
    mask_revision = df_dataset_final["banco_original"].astype(str).apply(
        lambda x: any(p in normalizar_busqueda(x) for p in patrones_revision)
    )

    df_revision_bancos = (
        df_dataset_final.loc[mask_revision, ["banco_original", "banco_estandarizado"]]
        .drop_duplicates()
        .sort_values(["banco_estandarizado", "banco_original"])
        .reset_index(drop=True)
    )

    print("Variantes revisadas:")
    display(df_revision_bancos)

    print("Resumen de nombres estandarizados revisados:")
    display(
        df_revision_bancos
        .groupby("banco_estandarizado", dropna=False)
        .agg(variantes=("banco_original", lambda s: " | ".join(sorted(set(map(str, s))))))
        .reset_index()
        .sort_values("banco_estandarizado")
    )
else:
    print("df_dataset_final está vacío o todavía no existe. Ejecuta primero el procesamiento y la depuración final.")

Variantes revisadas:


,banco_original,banco_estandarizado
0,BP D MIRO S.A.,D-MIRO
1,BP D-MIRO,D-MIRO
2,BP D-MIRO S.A.,D-MIRO
3,"BP BANCO DESARROLLO DE LOS PUEBLOS S.A., COD...",DESARROLLO DE LOS PUEBLOS S.A
4,BP BANCODESARROLLO,DESARROLLO DE LOS PUEBLOS S.A
5,BP DINERS,DINERS
6,BPDINERS,DINERS
7,BP FINCA,FINCA
8,BP FINCA S.A.,FINCA
9,BP VISIONFUND ECUADOR,VISIONFUND ECUADOR


Resumen de nombres estandarizados revisados:


,banco_estandarizado,variantes
0,D-MIRO,BP D MIRO S.A. | BP D-MIRO | BP D-MIRO S.A.
1,DESARROLLO DE LOS PUEBLOS S.A,"BP BANCO DESARROLLO DE LOS PUEBLOS S.A., COD..."
2,DINERS,BP DINERS | BPDINERS
3,FINCA,BP FINCA | BP FINCA S.A.
4,VISIONFUND ECUADOR,BP VISIONFUND ECUADOR | BP VISIONFUND ECUADOR ...


## 24. Construir dataset de periodos únicos

Este dataset sirve como fuente para filtros temporales del dashboard.

In [26]:
if df_dataset_final.empty:
    df_periodos = pd.DataFrame(columns=[
        "periodo", "fecha_corte", "anio", "mes", "numero_bancos", "numero_archivos", "numero_registros",
    ])
else:
    df_periodos = (
        df_dataset_final
        .dropna(subset=["periodo"])
        .groupby(["periodo", "fecha_corte"], dropna=False)
        .agg(
            numero_bancos=("banco_estandarizado", "nunique"),
            numero_archivos=("archivo_origen", "nunique"),
            numero_registros=("indicador", "size"),
        )
        .reset_index()
        .sort_values("periodo")
    )

    df_periodos["anio"] = df_periodos["periodo"].str.slice(0, 4).astype("Int64")
    df_periodos["mes"] = df_periodos["periodo"].str.slice(5, 7).astype("Int64")
    df_periodos = df_periodos[[
        "periodo", "fecha_corte", "anio", "mes", "numero_bancos", "numero_archivos", "numero_registros",
    ]]

print("Periodos únicos encontrados:", len(df_periodos))
display(df_periodos.head(120))

Periodos únicos encontrados: 205


,periodo,fecha_corte,anio,mes,numero_bancos,numero_archivos,numero_registros
0,2009-01,2009-01-31,2009,1,25,1,150
1,2009-02,2009-02-28,2009,2,24,1,144
2,2009-03,2009-03-31,2009,3,25,1,150
3,2009-04,2009-04-30,2009,4,25,1,150
4,2009-05,2009-05-31,2009,5,25,1,150
5,2009-06,2009-06-30,2009,6,25,1,150
6,2009-07,2009-07-31,2009,7,25,1,150
7,2009-08,2009-08-31,2009,8,25,1,150
8,2009-09,2009-09-30,2009,9,25,1,150
9,2009-10,2009-10-31,2009,10,25,1,150


## 25. Exportar datasets y auditorías

Se exportan CSV y Parquet en `01_datos_procesados`.

Archivo principal solicitado:

- `dataset_financiero_limpio.csv`
- `dataset_financiero_limpio.parquet`

Este dataset contiene únicamente:

```text
periodo | banco_estandarizado | indicador | valor | unidad | sentido
```

También se conserva una versión con trazabilidad para auditoría:

- `dataset_financiero_largo_auditoria.csv`
- `dataset_financiero_largo_auditoria.parquet`

Archivos auxiliares:

- `dataset_bancos.csv`
- `dataset_bancos.parquet`
- `dataset_periodos.csv`
- `dataset_periodos.parquet`

Auditorías:

- `auditoria_archivos.csv`
- `auditoria_eventos.csv`
- `matriz_control_indicadores.csv`

In [27]:
def guardar_csv_y_parquet(df, ruta_base: Path):
    """Guarda un DataFrame en CSV y Parquet usando la misma ruta base."""
    ruta_csv = ruta_base.with_suffix(".csv")
    ruta_parquet = ruta_base.with_suffix(".parquet")

    # CSV con utf-8-sig para que Excel reconozca tildes y ñ correctamente.
    df.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

    parquet_ok = True
    try:
        # Parquet es ideal para Streamlit/FastAPI porque ocupa menos y lee más rápido.
        df.to_parquet(ruta_parquet, index=False)
    except Exception as e:
        parquet_ok = False
        print(f"No se pudo guardar Parquet para {ruta_base.name}. Instala pyarrow si lo necesitas.")
        print("Detalle:", e)

    return ruta_csv, ruta_parquet if parquet_ok else None


rutas_generadas = []

# Dataset financiero principal solicitado: solo columnas limpias para análisis/dashboard.
rutas_generadas.extend([
    r for r in guardar_csv_y_parquet(
        df_dataset_financiero_limpio,
        PROCESSED_DIR / "dataset_financiero_limpio"
    ) if r
])

# Dataset financiero con trazabilidad. Útil para auditar de qué hoja/celda salió cada valor.
rutas_generadas.extend([
    r for r in guardar_csv_y_parquet(
        df_dataset_final,
        PROCESSED_DIR / "dataset_financiero_largo_auditoria"
    ) if r
])

# Dataset maestro de bancos.
rutas_generadas.extend([r for r in guardar_csv_y_parquet(df_bancos, PROCESSED_DIR / "dataset_bancos") if r])

# Dataset maestro de periodos.
rutas_generadas.extend([r for r in guardar_csv_y_parquet(df_periodos, PROCESSED_DIR / "dataset_periodos") if r])

# Auditorías en CSV.
ruta_auditoria_archivos = AUDIT_DIR / "auditoria_archivos.csv"
ruta_eventos = AUDIT_DIR / "auditoria_eventos.csv"
ruta_matriz = AUDIT_DIR / "matriz_control_indicadores.csv"

df_auditoria_archivos.to_csv(ruta_auditoria_archivos, index=False, encoding="utf-8-sig")
df_eventos.to_csv(ruta_eventos, index=False, encoding="utf-8-sig")

if "matriz_control" in globals() and isinstance(matriz_control, pd.DataFrame) and not matriz_control.empty:
    matriz_control.to_csv(ruta_matriz, index=False, encoding="utf-8-sig")
    rutas_generadas.append(ruta_matriz)

rutas_generadas.extend([ruta_auditoria_archivos, ruta_eventos])

print("Archivos generados:")
for ruta in rutas_generadas:
    print("-", ruta)

Archivos generados:
- C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\dataset_financiero_limpio.csv
- C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\dataset_financiero_limpio.parquet
- C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\dataset_financiero_largo_auditoria.csv
- C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\dataset_financiero_largo_auditoria.parquet
- C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\dataset_bancos.csv
- C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\dataset_bancos.parquet
- C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-b

## 26. Resumen final

Esta celda resume el estado general del proceso.

In [28]:
print("Resumen final")
print("=============")
print("Archivos Excel encontrados:", len(archivos_excel))
print("Archivos auditados:", len(df_auditoria_archivos))
print("Filas dataset de auditoría:", len(df_dataset_final))
print("Filas dataset_financiero_limpio:", len(df_dataset_financiero_limpio))
print("Bancos únicos:", len(df_bancos))
print("Periodos únicos:", len(df_periodos))
print("Eventos registrados:", len(df_eventos))

if not df_dataset_financiero_limpio.empty:
    print("\nIndicadores disponibles en dataset_financiero_limpio:")
    display(df_dataset_financiero_limpio.groupby("indicador").size().reset_index(name="filas"))

if not df_eventos.empty:
    print("\nEventos principales:")
    display(df_eventos.groupby("estado").size().reset_index(name="conteo").sort_values("conteo", ascending=False))

Resumen final
Archivos Excel encontrados: 205
Archivos auditados: 205
Filas dataset de auditoría: 29730
Filas dataset_financiero_limpio: 29730
Bancos únicos: 32
Periodos únicos: 205
Eventos registrados: 362

Indicadores disponibles en dataset_financiero_limpio:


,indicador,filas
0,activos_totales,4963
1,morosidad,4963
2,pasivos_totales,4963
3,patrimonio,4963
4,roe,4963
5,solvencia_proxy,4915



Eventos principales:


,estado,conteo
5,fallback_nombre_usado,197
2,duplicado_codigo_resuelto,65
3,duplicado_nombre_resuelto,52
1,duplicado_balance_resuelto,40
0,componente_fk_no_encontrado,2
4,fallback_fi_nombre_usado,2
6,indicador_financiero_no_encontrado_final,2
7,solvencia_proxy_no_encontrada,2


## 27. Siguiente paso recomendado

Cuando este notebook genere correctamente el dataset largo, el siguiente paso es construir la capa analítica para el dashboard:

1. Ranking por activos.
2. Ranking por ROE.
3. Ranking por menor morosidad.
4. Ranking por solvencia proxy.
5. Evolución histórica por banco.
6. Comparador entre bancos.

La estructura final ya queda lista para Streamlit porque usa formato largo:

```text
periodo | banco_estandarizado | indicador | valor
```